# Tempo até a reversa após a entrega ao cliente

**Pergunta de negócio:** depois que o pedido chega na casa do cliente, quanto tempo ele leva para *abrir uma reversa* (troca **ou** devolução)?

## Metodologia (rastreável)

| Elemento | Fonte | Campo |
|---|---|---|
| Chegada na casa do cliente | `insider-lake-sensitive.integrated_br.shippings_br` | `delivered_date` (DATETIME) — 1ª entrega por pedido (`MIN`) |
| Abertura da reversa | `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br` | `created_at` (TIMESTAMP) → convertido p/ `America/Sao_Paulo` |
| Tipo de reversa | idem Troquecommerce | `reverse_type` (Troca · Devolução · Troca e devolução · Sem Reembolso) |
| Pedidos válidos (filtro T&D) | `insider-data-lake.business.insider_orders` | `paid`, `is_cancelled = FALSE`, exclusão cupons TF-/TFIN/IR/Item errado, lojas `insider-world` + `insider-store-loja` |

**Chave de join:** `order_name` (formato `IN-XXXXXXX`, idêntico nas três tabelas — validado 2026-07-27).

**Métrica:** `dias_ate_reversa = created_at(BR) − delivered_date`, em dias fracionários. Grão = **uma linha por reversa** (`order_name × id_reversa`, dedup por `updated_at DESC`).

**Janela:** entregas nos **últimos 12 meses** (âncora = mês de entrega).

### Limitações declaradas
- **Censura à direita:** meses de entrega recentes ainda não tiveram tempo de acumular reversas tardias — a mediana/volume dos últimos ~30 dias é subestimada. Sinalizado nos gráficos de coorte.
- **Reversas antes da entrega (`dias < 0`, ~3–4%):** ruído de `delivered_date` (multi-volume, data de entrega imprecisa) ou reversa atrelada a outra remessa. Excluídas das estatísticas de tempo e reportadas à parte.
- Fonte da reversa é a **plataforma Troquecommerce** (quando o cliente *abriu* a reversa) — não confundir com `order_refunds_br` (evento financeiro no Shopify, universo maior).


## 0. Setup e parâmetros

In [32]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_ID = "insider-data-lake"
client = bigquery.Client(project=PROJECT_ID)

# Janela: últimos 12 meses a partir de hoje (âncora = data de entrega)
HOJE = pd.Timestamp.today().normalize()
REF_DATE = (HOJE - pd.DateOffset(months=12)).date().isoformat()
DATA_TAG = HOJE.strftime("%Y%m%d")
OUT_DIR = "../../outputs"

print(f"Conectado a {PROJECT_ID}")

print(f"Janela de entregas: delivered_date >= {REF_DATE}  (hoje = {HOJE.date()})")

/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Conectado a insider-data-lake
Janela de entregas: delivered_date >= 2025-08-05  (hoje = 2026-08-05)


# Parte A — Tempo ENTREGA → reversa

*Do momento em que o pedido chega na casa do cliente (`delivered_date`) até a abertura da reversa.*

## 1. Extração — reversa × entrega (grão: uma linha por reversa)

In [33]:
query = f"""
WITH orders_validos AS (
  SELECT DISTINCT order_name
  FROM `insider-data-lake.business.insider_orders`
  WHERE order_status = 'paid' AND is_cancelled = FALSE
    AND (coupon_code IS NULL OR (
      NOT STARTS_WITH(coupon_code, 'TF-')  AND NOT STARTS_WITH(coupon_code, 'TFIN')
      AND NOT STARTS_WITH(coupon_code, 'IR') AND NOT coupon_code LIKE '%Item errado%'))
    AND store IN ('shopify_insider-world', 'shopify_insider-store-loja')
),

-- 1ª entrega ao cliente por pedido
entrega AS (
  SELECT order_name, MIN(delivered_date) AS delivered_date
  FROM `insider-lake-sensitive.integrated_br.shippings_br`
  WHERE delivered_date IS NOT NULL
  GROUP BY order_name
),

-- reversa deduplicada por (order_name, id_reversa) via updated_at DESC
reversa AS (
  SELECT order_name, id_reversa, created_at, reverse_type
  FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br`
  WHERE status <> 'Cancelado' AND created_at IS NOT NULL AND id_reversa IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (PARTITION BY order_name, id_reversa ORDER BY updated_at DESC) = 1
)

SELECT
  r.order_name,
  r.id_reversa,
  r.reverse_type,
  e.delivered_date,
  DATETIME(r.created_at, 'America/Sao_Paulo')                                             AS reversa_criada_em,
  DATETIME_DIFF(DATETIME(r.created_at, 'America/Sao_Paulo'), e.delivered_date, HOUR)/24.0  AS dias_ate_reversa
FROM reversa r
JOIN entrega e USING (order_name)
JOIN orders_validos o USING (order_name)
WHERE e.delivered_date >= '{REF_DATE}'
"""

df = client.query(query).to_dataframe(create_bqstorage_client=False)
print(f"{len(df):,} reversas casadas com entrega (pedidos válidos, últimos 12 meses)")
df.head()


112,898 reversas casadas com entrega (pedidos válidos, últimos 12 meses)


,order_name,id_reversa,reverse_type,delivered_date,reversa_criada_em,dias_ate_reversa
0,IN-4034538,abdcec5e-d791-4c74-9081-8eb1035ef81f,Troca,2026-06-08 14:17:00,2026-06-17 08:17:34.457934,8.75
1,IN-4038898,5c769cf5-34e6-4b9f-8585-a1e651bea00a,Troca,2026-06-08 19:50:00,2026-06-15 11:45:14.406782,6.67
2,IN-4039175,e983413d-c2c5-4ffb-baee-2a4e2b9e85af,Troca,2026-06-08 14:09:59,2026-06-13 23:22:55.078702,5.38
3,IN-4050624,e484b89c-e8af-4343-853e-b8be817a4964,Troca,2026-06-08 14:42:00,2026-06-11 03:48:49.055352,2.54
4,IN-4051932,938214ea-15de-4545-83a6-ec7a6f6b7048,Troca,2026-06-08 16:43:00,2026-06-10 22:25:33.343107,2.25


## 2. Qualidade dos dados e enriquecimento

Excluímos `dias_ate_reversa < 0` das estatísticas de tempo (ruído de `delivered_date`) e criamos: mês de entrega, tipo simplificado e bucket de tempo.

In [34]:
# Enriquecimento
df["mes_entrega"] = pd.to_datetime(df["delivered_date"]).dt.to_period("M").astype(str)

# tipo simplificado (Troca / Devolução / Mista / Sem reembolso)
mapa_tipo = {
    "Troca": "Troca",
    "Devolução": "Devolução",
    "Troca e devolução": "Mista",
    "Sem Reembolso": "Sem reembolso",
}
df["tipo"] = df["reverse_type"].map(mapa_tipo).fillna("Outros")

# buckets de tempo
bins = [0, 2, 7, 15, 30, np.inf]
labels = ["0–2d", "3–7d", "8–15d", "16–30d", "31d+"]
df["bucket"] = pd.cut(df["dias_ate_reversa"], bins=bins, labels=labels, include_lowest=True)

# QA
n_total = len(df)
n_neg = int((df["dias_ate_reversa"] < 0).sum())
print(f"Total de reversas casadas : {n_total:,}")
print(f"Reversas antes da entrega : {n_neg:,}  ({n_neg/n_total:.1%}) -> excluídas do tempo")

# base válida para estatísticas de tempo
dfv = df[df["dias_ate_reversa"] >= 0].copy()
print(f"Base válida (dias >= 0)   : {len(dfv):,}")
df["reverse_type"].value_counts()


Total de reversas casadas : 112,898
Reversas antes da entrega : 4,132  (3.7%) -> excluídas do tempo
Base válida (dias >= 0)   : 108,766


reverse_type
Troca                99243
Devolução            12128
Sem Reembolso         1223
Troca e devolução      304
Name: count, dtype: int64

## 3. Distribuição geral — quantos dias até abrir a reversa

In [35]:
pcts = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
resumo_geral = dfv["dias_ate_reversa"].describe(percentiles=pcts).round(2)
print(resumo_geral)

# marcos práticos
for d in [1, 2, 7, 15, 30]:
    frac = (dfv["dias_ate_reversa"] <= d).mean()
    print(f"% das reversas abertas em até {d:>2}d após a entrega: {frac:.1%}")

# histograma interativo (cap visual em 60d para leitura)
cap = dfv["dias_ate_reversa"].clip(upper=60)
med = dfv["dias_ate_reversa"].median()

fig = px.histogram(cap, nbins=60, title="Dias entre entrega e abertura da reversa (cap visual 60d)",
                    labels={"value": "dias até a reversa"}, color_discrete_sequence=["#2b6cb0"])
fig.add_vline(x=med, line_dash="dash", line_color="#e53e3e",
              annotation_text=f"mediana = {med:.1f}d", annotation_position="top right")
fig.update_layout(xaxis_title="dias até a reversa", yaxis_title="nº de reversas", showlegend=False,
                   bargap=0.02, height=450)

fig.show()

count   108,766.00
mean          6.04
std          13.64
min           0.00
10%           0.12
25%           0.54
50%           2.12
75%           6.04
90%          13.79
95%          21.42
max         309.83
Name: dias_ate_reversa, dtype: float64
% das reversas abertas em até  1d após a entrega: 35.6%
% das reversas abertas em até  2d após a entrega: 48.6%
% das reversas abertas em até  7d após a entrega: 78.9%
% das reversas abertas em até 15d após a entrega: 91.2%
% das reversas abertas em até 30d após a entrega: 96.9%


## 3.1 Pareto — dias até a reversa vs. % acumulado coberto

A partir de quantos dias após a entrega já cobrimos 80% das reversas?


In [36]:
MAX_DIAS_PARETO = 60  # janela de leitura do pareto (dias)

dias_int = dfv["dias_ate_reversa"].clip(lower=0).apply(np.floor).astype(int)
contagem_dia = dias_int.value_counts().sort_index()
contagem_dia = contagem_dia.reindex(range(0, contagem_dia.index.max() + 1), fill_value=0)

pareto = contagem_dia.reset_index()
pareto.columns = ["dia", "n_reversas"]
pareto["pct_acumulado"] = pareto["n_reversas"].cumsum() / pareto["n_reversas"].sum() * 100

# dia em que se atinge >=80% (e outros marcos úteis)
marcos_pareto = {}
for alvo in [50, 80, 90, 95]:
    linha = pareto[pareto["pct_acumulado"] >= alvo].iloc[0]
    marcos_pareto[alvo] = int(linha["dia"])
    print(f">= {alvo}% das reversas cobertas a partir de {int(linha['dia'])} dias após a entrega")

dia_80 = marcos_pareto[80]

pareto_plot = pareto[pareto["dia"] <= MAX_DIAS_PARETO].copy()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=pareto_plot["dia"], y=pareto_plot["n_reversas"], name="nº reversas (dia)",
                      marker_color="#2b6cb0", opacity=0.6), secondary_y=False)
fig.add_trace(go.Scatter(x=pareto_plot["dia"], y=pareto_plot["pct_acumulado"], name="% acumulado",
                          mode="lines+markers", line=dict(color="#e53e3e")), secondary_y=True)
fig.add_hline(y=80, line_dash="dash", line_color="#e53e3e", secondary_y=True,
              annotation_text="80%", annotation_position="top left")
fig.add_vline(x=dia_80, line_dash="dot", line_color="#38a169",
              annotation_text=f"dia {dia_80}", annotation_position="top")
fig.update_yaxes(title_text="nº de reversas no dia", secondary_y=False)
fig.update_yaxes(title_text="% acumulado", range=[0, 100], secondary_y=True)
fig.update_xaxes(title_text="dias até a reversa (após entrega)")
fig.update_layout(title=f"Pareto — cobertura acumulada de reversas por dia (80% atingido no dia {dia_80})",
                   height=450)
fig.show()


>= 50% das reversas cobertas a partir de 2 dias após a entrega
>= 80% das reversas cobertas a partir de 7 dias após a entrega
>= 90% das reversas cobertas a partir de 13 dias após a entrega
>= 95% das reversas cobertas a partir de 21 dias após a entrega


## 4. Troca vs Devolução

In [37]:
stats_tipo = (dfv.groupby("tipo")["dias_ate_reversa"]
    .agg(n="count",
         mediana="median",
         media="mean",
         p90=lambda s: s.quantile(0.90),
         pct_ate_7d=lambda s: (s <= 7).mean())
    .sort_values("n", ascending=False)
    .round(2))
display(stats_tipo)

# boxplot interativo (troca vs devolução)
foco = dfv[dfv["tipo"].isin(["Troca", "Devolução"])].copy()
foco["dias_cap"] = foco["dias_ate_reversa"].clip(upper=60)

fig = px.box(foco, x="dias_cap", y="tipo", orientation="h", points=False,
             title="Distribuição de dias até a reversa — Troca vs Devolução (cap 60d)",
             labels={"dias_cap": "dias até a reversa", "tipo": ""},
             color="tipo", color_discrete_sequence=["#2b6cb0", "#e53e3e"])
fig.update_layout(showlegend=False, height=400)

fig.show()

,n,mediana,media,p90,pct_ate_7d
tipo,,,,,
Troca,95600,2.12,6.24,14.21,0.78
Devolução,11698,2.04,4.11,8.96,0.86
Sem reembolso,1178,2.79,8.80,18.89,0.73
Mista,290,2.12,3.71,9.30,0.85


## 5. Buckets de tempo (0–2d · 3–7d · 8–15d · 16–30d · 31d+)

In [38]:
# distribuição geral por bucket
dist_bucket = (dfv["bucket"].value_counts(normalize=True).reindex(labels) * 100).round(1)
tab_bucket = pd.DataFrame({"pct_%": dist_bucket, "n": dfv["bucket"].value_counts().reindex(labels)})
display(tab_bucket)

# crosstab % por tipo (Troca / Devolução)
ct = pd.crosstab(dfv["tipo"], dfv["bucket"], normalize="index")[labels] * 100
ct = ct.loc[[t for t in ["Troca", "Devolução", "Mista", "Sem reembolso"] if t in ct.index]].round(1)
display(ct)

# barra empilhada interativa
ct_long = ct.reset_index().melt(id_vars="tipo", var_name="bucket", value_name="pct")
fig = px.bar(ct_long, x="pct", y="tipo", color="bucket", orientation="h",
             title="Composição dos buckets de tempo por tipo de reversa (%)",
             labels={"pct": "% das reversas do tipo", "tipo": "", "bucket": "bucket"},
             category_orders={"bucket": labels}, color_discrete_sequence=px.colors.sequential.Blues_r,
             text=ct_long["pct"].map(lambda v: f"{v:.1f}%"))

fig.update_layout(barmode="stack", height=400)
fig.show()

,pct_%,n
bucket,,
0–2d,48.60,52815
3–7d,30.30,32970
8–15d,12.30,13386
16–30d,5.70,6241
31d+,3.10,3354


bucket,0–2d,3–7d,8–15d,16–30d,31d+
tipo,,,,,
Troca,48.50,29.60,12.60,6.10,3.30
Devolução,49.90,36.00,10.00,3.10,1.00
Mista,48.60,36.20,11.00,4.10,0.00
Sem reembolso,43.20,29.50,14.90,6.20,6.30


## 6. Coorte mensal (por mês de entrega)

⚠️ **Censura à direita:** os meses mais recentes ainda não tiveram tempo de acumular reversas tardias — mediana e volume subestimados no fim da série.

In [39]:
coorte = (dfv.groupby("mes_entrega")
    .agg(n_reversas=("id_reversa", "count"),
         mediana_dias=("dias_ate_reversa", "median"),
         pct_ate_7d=("dias_ate_reversa", lambda s: (s <= 7).mean() * 100),
         pct_ate_15d=("dias_ate_reversa", lambda s: (s <= 15).mean() * 100))
    .round(2))
display(coorte)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=coorte.index, y=coorte["n_reversas"], name="nº reversas",
                      marker_color="#718096", opacity=0.35), secondary_y=True)
fig.add_trace(go.Scatter(x=coorte.index, y=coorte["mediana_dias"], name="mediana (dias)",
                          mode="lines+markers", marker=dict(color="#2b6cb0"), line=dict(color="#2b6cb0")),
              secondary_y=False)
fig.update_yaxes(title_text="mediana de dias até a reversa", secondary_y=False)
fig.update_yaxes(title_text="nº de reversas", secondary_y=True)
fig.update_xaxes(tickangle=45)
fig.update_layout(title="Coorte por mês de entrega — mediana de dias até a reversa e volume", height=450)

fig.show()

,n_reversas,mediana_dias,pct_ate_7d,pct_ate_15d
mes_entrega,,,,
2025-08,7131,2.33,79.33,91.24
2025-09,7810,2.08,79.80,91.18
2025-10,8387,2.04,80.39,91.82
2025-11,13483,2.00,80.13,91.21
2025-12,16098,3.58,67.19,84.02
2026-01,8173,2.04,79.51,91.21
2026-02,7498,2.00,78.99,91.70
2026-03,13904,1.88,82.13,92.98
2026-04,8179,2.08,79.92,92.63


# Parte B — Tempo COMPRA → reversa

*Da **data da compra** (`processed_at` → `America/Sao_Paulo`) até a abertura da reversa.*
População = todas as reversas válidas com compra casada por `order_name`; janela = **compras dos últimos 12 meses**. Validado (2026-07-27): **0 reversas antes da compra** e **99,4%** também têm entrega casada — as duas populações praticamente coincidem.

## 7. Extração — compra × reversa

In [40]:
query_compra = f"""
WITH compras AS (
  SELECT DISTINCT
    order_name,
    DATE(TIMESTAMP(processed_at), 'America/Sao_Paulo') AS data_compra
  FROM `insider-data-lake.business.insider_orders`
  WHERE order_status = 'paid' AND is_cancelled = FALSE
    AND (coupon_code IS NULL OR (
      NOT STARTS_WITH(coupon_code, 'TF-')  AND NOT STARTS_WITH(coupon_code, 'TFIN')
      AND NOT STARTS_WITH(coupon_code, 'IR') AND NOT coupon_code LIKE '%Item errado%'))
    AND order_name IS NOT NULL AND processed_at IS NOT NULL
    AND store IN ('shopify_insider-world', 'shopify_insider-store-loja')
),

reversa AS (
  SELECT order_name, id_reversa, created_at, reverse_type
  FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br`
  WHERE status <> 'Cancelado' AND created_at IS NOT NULL AND id_reversa IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (PARTITION BY order_name, id_reversa ORDER BY updated_at DESC) = 1
),

entrega AS (
  SELECT order_name, MIN(delivered_date) AS delivered_date
  FROM `insider-lake-sensitive.integrated_br.shippings_br`
  WHERE delivered_date IS NOT NULL
  GROUP BY order_name
)

SELECT
  r.order_name,
  r.id_reversa,
  r.reverse_type,
  c.data_compra,
  DATE(e.delivered_date)                                             AS data_entrega,
  DATE(r.created_at, 'America/Sao_Paulo')                            AS data_reversa,
  DATE_DIFF(DATE(r.created_at, 'America/Sao_Paulo'), c.data_compra, DAY) AS dias_compra_ate_reversa
FROM reversa r
JOIN compras c USING (order_name)
LEFT JOIN entrega e USING (order_name)
WHERE c.data_compra >= '{REF_DATE}'
"""

df_compra = client.query(query_compra).to_dataframe(create_bqstorage_client=False)

# enriquecimento
df_compra["mes_compra"] = pd.to_datetime(df_compra["data_compra"]).dt.to_period("M").astype(str)
df_compra["tipo"] = df_compra["reverse_type"].map(mapa_tipo).fillna("Outros")
bins_c = [0, 7, 15, 30, 60, np.inf]
labels_c = ["0–7d", "8–15d", "16–30d", "31–60d", "61d+"]
df_compra["bucket"] = pd.cut(df_compra["dias_compra_ate_reversa"], bins=bins_c, labels=labels_c, include_lowest=True)

n = len(df_compra); neg = int((df_compra["dias_compra_ate_reversa"] < 0).sum())
tem_ent = df_compra["data_entrega"].notna().mean()
print(f"Reversas com compra casada : {n:,}")
print(f"Negativas (reversa < compra): {neg:,}  ({neg/n:.2%})")
print(f"Tambem com entrega casada  : {tem_ent:.1%}")

dfc = df_compra[df_compra["dias_compra_ate_reversa"] >= 0].copy()
df_compra.head()


Reversas com compra casada : 110,934
Negativas (reversa < compra): 0  (0.00%)
Tambem com entrega casada  : 99.4%


,order_name,id_reversa,reverse_type,data_compra,data_entrega,data_reversa,dias_compra_ate_reversa,mes_compra,tipo,bucket
0,IN-3125435-ENTR-0126,093f028b-904e-4bd1-8d8e-cbfefdbd353d,Troca,2025-11-12,2025-12-15,2025-12-16,34,2025-11,Troca,31–60d
1,IN-3293157,cdcf01d0-abd7-4550-a0ec-14b6e0763cf5,Troca,2025-11-30,2025-12-08,2026-01-07,38,2025-11,Troca,31–60d
2,IN-3401259,add28aa9-c55e-4524-ac38-7e190c1e4e79,Troca,2025-12-15,2025-12-22,2026-01-13,29,2025-12,Troca,16–30d
3,IN-3126376-ENTR-0126,dbc75a7a-ac78-45a7-8846-d5f01ae35603,Troca,2025-11-13,2026-01-09,2026-01-09,57,2025-11,Troca,31–60d
4,IN-3333214,af561329-6d86-4afd-b8d0-4f95e312e2a0,Troca,2025-12-05,2025-12-18,2026-01-06,32,2025-12,Troca,31–60d


## 8. Distribuição e tipo (compra → reversa)

In [41]:
pcts = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
print(dfc["dias_compra_ate_reversa"].describe(percentiles=pcts).round(2))
for d in [7, 15, 30, 60]:
    print(f"% das reversas abertas em até {d:>2}d após a compra: {(dfc['dias_compra_ate_reversa'] <= d).mean():.1%}")

# histograma interativo (cap visual em 90d para leitura)
cap = dfc["dias_compra_ate_reversa"].clip(upper=90)
med = dfc["dias_compra_ate_reversa"].median()

fig = px.histogram(cap, nbins=60, title="Dias entre compra e abertura da reversa (cap visual 90d)",
                    labels={"value": "dias após a compra"}, color_discrete_sequence=["#2f855a"])
fig.add_vline(x=med, line_dash="dash", line_color="#e53e3e",
              annotation_text=f"mediana = {med:.0f}d", annotation_position="top right")
fig.update_layout(xaxis_title="dias após a compra", yaxis_title="nº de reversas", showlegend=False,
                   bargap=0.02, height=450)
fig.show()

stats_tipo_compra = (dfc.groupby("tipo")["dias_compra_ate_reversa"]
    .agg(n="count", mediana="median", media="mean",
         p90=lambda s: s.quantile(0.90),
         pct_ate_15d=lambda s: (s <= 15).mean())
    .sort_values("n", ascending=False).round(2))
display(stats_tipo_compra)


count   110,934.00
mean         12.88
std          16.23
min           0.00
10%           4.00
25%           6.00
50%           8.00
75%          14.00
90%          23.00
95%          35.00
max         314.00
Name: dias_compra_ate_reversa, dtype: Float64
% das reversas abertas em até  7d após a compra: 42.4%
% das reversas abertas em até 15d após a compra: 79.9%
% das reversas abertas em até 30d após a compra: 94.0%
% das reversas abertas em até 60d após a compra: 97.8%


,n,mediana,media,p90,pct_ate_15d
tipo,,,,,
Troca,97373,8.00,12.99,24,0.79
Devolução,12060,8.00,11.88,20,0.84
Sem reembolso,1196,9.00,14.70,26,0.78
Mista,305,9.00,10.86,18,0.86


## 8.1 Pareto — dias até a reversa (compra) vs. % acumulado coberto

A partir de quantos dias após a compra já cobrimos 80% das reversas?


In [42]:
MAX_DIAS_PARETO_COMPRA = 90  # janela de leitura do pareto (dias)

dias_int_c = dfc["dias_compra_ate_reversa"].clip(lower=0).apply(np.floor).astype(int)
contagem_dia_c = dias_int_c.value_counts().sort_index()
contagem_dia_c = contagem_dia_c.reindex(range(0, contagem_dia_c.index.max() + 1), fill_value=0)

pareto_compra = contagem_dia_c.reset_index()
pareto_compra.columns = ["dia", "n_reversas"]
pareto_compra["pct_acumulado"] = pareto_compra["n_reversas"].cumsum() / pareto_compra["n_reversas"].sum() * 100

# dia em que se atinge >=80% (e outros marcos úteis)
marcos_pareto_compra = {}
for alvo in [50, 80, 90, 95]:
    linha = pareto_compra[pareto_compra["pct_acumulado"] >= alvo].iloc[0]
    marcos_pareto_compra[alvo] = int(linha["dia"])
    print(f">= {alvo}% das reversas cobertas a partir de {int(linha['dia'])} dias após a compra")

dia_80_compra = marcos_pareto_compra[80]

pareto_compra_plot = pareto_compra[pareto_compra["dia"] <= MAX_DIAS_PARETO_COMPRA].copy()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=pareto_compra_plot["dia"], y=pareto_compra_plot["n_reversas"], name="nº reversas (dia)",
                      marker_color="#2f855a", opacity=0.6), secondary_y=False)
fig.add_trace(go.Scatter(x=pareto_compra_plot["dia"], y=pareto_compra_plot["pct_acumulado"], name="% acumulado",
                          mode="lines+markers", line=dict(color="#e53e3e")), secondary_y=True)
fig.add_hline(y=80, line_dash="dash", line_color="#e53e3e", secondary_y=True,
              annotation_text="80%", annotation_position="top left")
fig.add_vline(x=dia_80_compra, line_dash="dot", line_color="#2b6cb0",
              annotation_text=f"dia {dia_80_compra}", annotation_position="top")
fig.update_yaxes(title_text="nº de reversas no dia", secondary_y=False)
fig.update_yaxes(title_text="% acumulado", range=[0, 100], secondary_y=True)
fig.update_xaxes(title_text="dias até a reversa (após compra)")
fig.update_layout(title=f"Pareto — cobertura acumulada de reversas por dia após a compra (80% atingido no dia {dia_80_compra})",
                   height=450)
fig.show()


>= 50% das reversas cobertas a partir de 8 dias após a compra
>= 80% das reversas cobertas a partir de 16 dias após a compra
>= 90% das reversas cobertas a partir de 23 dias após a compra
>= 95% das reversas cobertas a partir de 35 dias após a compra


## 9. Buckets e coorte mensal (por mês de compra)

⚠️ **Censura à direita:** meses de compra recentes ainda não tiveram tempo de gerar reversas tardias — subestimados no fim da série.

In [43]:
tab_bucket_compra = pd.DataFrame({
    "pct_%": (dfc["bucket"].value_counts(normalize=True).reindex(labels_c) * 100).round(1),
    "n": dfc["bucket"].value_counts().reindex(labels_c),
})
display(tab_bucket_compra)

coorte_compra = (dfc.groupby("mes_compra")
    .agg(n_reversas=("id_reversa", "count"),
         mediana_dias=("dias_compra_ate_reversa", "median"),
         pct_ate_15d=("dias_compra_ate_reversa", lambda s: (s <= 15).mean() * 100),
         pct_ate_30d=("dias_compra_ate_reversa", lambda s: (s <= 30).mean() * 100))
    .round(2))
display(coorte_compra)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=coorte_compra.index, y=coorte_compra["n_reversas"], name="nº reversas",
                      marker_color="#718096", opacity=0.35), secondary_y=True)
fig.add_trace(go.Scatter(x=coorte_compra.index, y=coorte_compra["mediana_dias"], name="mediana (dias)",
                          mode="lines+markers", marker=dict(color="#2f855a"), line=dict(color="#2f855a")),
              secondary_y=False)
fig.update_yaxes(title_text="mediana dias compra→reversa", secondary_y=False)
fig.update_yaxes(title_text="nº de reversas", secondary_y=True)
fig.update_xaxes(tickangle=45)
fig.update_layout(title="Coorte por mês de compra — mediana compra→reversa e volume", height=450)
fig.show()


,pct_%,n
bucket,,
0–7d,42.40,47072
8–15d,37.50,41616
16–30d,14.00,15579
31–60d,3.90,4272
61d+,2.20,2395


,n_reversas,mediana_dias,pct_ate_15d,pct_ate_30d
mes_compra,,,,
2025-08,6925,9.00,78.74,92.09
2025-09,7720,8.00,80.82,92.46
2025-10,9344,9.00,81.54,94.31
2025-11,19747,10.00,74.98,91.84
2025-12,11463,12.00,64.09,90.47
2026-01,7814,7.00,82.95,93.29
2026-02,8289,8.00,81.57,94.88
2026-03,14812,7.00,86.15,96.73
2026-04,7279,8.00,82.15,94.90


## 10. Decomposição compra → entrega → reversa

Nas reversas com entrega casada, o tempo total compra→reversa se divide em **envio** (compra→entrega) e **decisão** (entrega→reversa). Medianas não somam exatamente (distribuições diferentes).

In [44]:
dec = df_compra[df_compra["data_entrega"].notna()].copy()
dec["dias_compra_entrega"] = (pd.to_datetime(dec["data_entrega"]) - pd.to_datetime(dec["data_compra"])).dt.days
dec["dias_entrega_reversa"] = (pd.to_datetime(dec["data_reversa"]) - pd.to_datetime(dec["data_entrega"])).dt.days
dec_v = dec[(dec["dias_compra_entrega"] >= 0) & (dec["dias_entrega_reversa"] >= 0) & (dec["dias_compra_ate_reversa"] >= 0)]

tab_dec = pd.DataFrame({
    "etapa": ["compra -> entrega (envio)", "entrega -> reversa (decisao)", "compra -> reversa (total)"],
    "mediana_dias": [dec_v["dias_compra_entrega"].median(),
                     dec_v["dias_entrega_reversa"].median(),
                     dec_v["dias_compra_ate_reversa"].median()],
    "media_dias": [dec_v["dias_compra_entrega"].mean(),
                   dec_v["dias_entrega_reversa"].mean(),
                   dec_v["dias_compra_ate_reversa"].mean()],
}).round(1)
display(tab_dec)
print(f"Base decomposicao (3 marcos casados, nao-negativos): {len(dec_v):,}")


,etapa,mediana_dias,media_dias
0,compra -> entrega (envio),5.00,7.10
1,entrega -> reversa (decisao),2.00,5.80
2,compra -> reversa (total),8.00,12.90


Base decomposicao (3 marcos casados, nao-negativos): 108,690


# Parte C — OKR de Troca & Devolução (Source of Truth · COHORT)

> **Fonte da verdade (SoT):** métrica **por coorte de compra**. A reversa é atribuída ao **mês/semana da compra** do pedido, via `order_name`. Numerador **lifetime** (todas as reversas do pedido, sem corte por data).
> - **Numerador:** `SUM(return_quantity)` de **todas** as reversas não canceladas — **mesmo universo de reversa da definição mensal legada** (ver Parte D), sem exigir que o pedido de origem seja `paid`/não-cancelado/loja elegível e sem exigir que o SKU revertido conste como item comprado. A única exigência é existir algum pedido com `processed_at` para ancorar a data de compra (cobertura medida: **99,98%** do universo de reversa).
> - **Denominador:** `SUM(quantity)` de itens comprados no período (`paid`/filtros de cupom/lojas) — inalterado. Reconcilia em **zero** com o legado.
> - `T&D% = reversas_do_cohort ÷ vendas_do_cohort × 100`.
>
> ⚠️ **Dedup, como está no código:** `SELECT DISTINCT (order_name, id_reversa, sku, return_quantity)` — `return_quantity` está **dentro** do DISTINCT. Duas linhas da mesma chave com quantidades diferentes sobrevivem e são ambas somadas. O legado tem o **mesmo** construto (simétrico), mas é exposição real a dupla contagem — instrumentada na Parte D e registrada como dívida em `AUDITORIA_PREMISSAS_OKR_TD.md` §6.1.

**Por que essa definição:** o cohort e a métrica legada (visão mensal por fluxo, Parte D) enxergam **exatamente o mesmo conjunto de reversas**; a única diferença estrutural entre as duas é a **ancoragem de data** — mês da compra (cohort) vs. mês de abertura da reversa (legado). Antes da revisão de 2026-08-04, o cohort também exigia pedido válido e SKU casado no numerador, o que introduzia uma segunda diferença (assimetria de universo) misturada com o efeito de ancoragem.

**Evidência do impacto da ancoragem:** **44% do numerador legado de jan/26** (10.151 de 23.048 unidades) vem de compras de **2025** — a safra de Black Friday chegando à janela de troca em janeiro, dividida pelas vendas de janeiro. O legado lê **10,58%**; o cohort lê **8,42%** e devolve aquelas unidades para nov/dez-25. Detalhamento completo em `AUDITORIA_PREMISSAS_OKR_TD.md`.

**Onde a métrica vive (auditável, fora do notebook):**
| Arquivo | O quê |
|---|---|
| [`sql/td_cohort_mensal.sql`](sql/td_cohort_mensal.sql) | KR mensal (MBR), com `maduro_em` + `status` na própria query |
| [`sql/td_cohort_semanal.sql`](sql/td_cohort_semanal.sql) | HM semanal, com `reportavel_em` + `status` + `is_w2` + média móvel 4 sem |
| [`AUDITORIA_PREMISSAS_OKR_TD.md`](AUDITORIA_PREMISSAS_OKR_TD.md) | confronto premissa a premissa vs. legado + ponte de numerador |

As duas queries são **gêmeas estruturais** da legada (mesmos nomes e ordem de CTEs), para que o diff lado a lado mostre apenas **premissa**, não estilo. Janela alinhada ao legado: `start_date = 2026-01-01`, até o último período fechado.

**Definições + maturação (agora dentro do SQL, não no pandas):**
- **KR mensal (MBR):** mês só é `oficial (maduro)` no **dia 15 de M+1**. Meses recentes = `em maturação` — leitura de tendência, nunca de resultado.
- **HM semanal:** semana reportável a partir de **W-2** (`semana_ini + 21 dias`). Semanas mais recentes estão subestimadas por **censura à direita**, não por melhora operacional.


## 11. Extração diária do cohort (base p/ KR mensal e HM semanal)

In [45]:
query_cohort = """
WITH compras_filtradas AS (
  SELECT DISTINCT order_id, order_name, DATE(TIMESTAMP(processed_at),'America/Sao_Paulo') AS data_compra
  FROM `insider-data-lake.business.insider_orders`
  WHERE order_status='paid' AND is_cancelled=FALSE
    AND (coupon_code IS NULL OR (NOT STARTS_WITH(coupon_code,'TF-') AND NOT STARTS_WITH(coupon_code,'TFIN')
      AND NOT STARTS_WITH(coupon_code,'IR') AND NOT coupon_code LIKE '%Item errado%'))
    AND order_name IS NOT NULL AND processed_at IS NOT NULL
    AND store IN ('shopify_insider-world','shopify_insider-store-loja')),
itens_comprados AS (
  SELECT c.data_compra, c.order_name, i.sku, SUM(i.quantity) AS qt_comprados
  FROM compras_filtradas c JOIN `insider-data-lake.business.insider_order_items` i ON c.order_id=i.order_id
  WHERE i.sku IS NOT NULL AND c.data_compra >= '2025-01-01' GROUP BY 1,2,3),
vendas_por_dia AS (
  SELECT data_compra AS dia, SUM(qt_comprados) AS vendas FROM itens_comprados GROUP BY 1),
-- data de compra de QUALQUER pedido (sem exigir paid/loja/cupom) — mesmo universo de pedido que a
-- definição legada aceita implicitamente (ela nunca valida o pedido da reversa). Só ancora a data.
todos_pedidos AS (
  SELECT order_name, MIN(DATE(TIMESTAMP(processed_at),'America/Sao_Paulo')) AS data_compra
  FROM `insider-data-lake.business.insider_orders`
  WHERE order_name IS NOT NULL AND processed_at IS NOT NULL
  GROUP BY 1),
-- universo de reversa IDÊNTICO ao da definição legada: sem exigir pedido válido nem SKU casado ao item comprado.
reversas_universo AS (
  SELECT DISTINCT order_name, id_reversa, sku, SAFE_CAST(return_quantity AS FLOAT64) AS return_quantity
  FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br`
  WHERE status<>'Cancelado' AND order_name IS NOT NULL AND sku IS NOT NULL AND id_reversa IS NOT NULL
    AND created_at IS NOT NULL),
reversas_por_pedido AS (
  SELECT order_name, SUM(return_quantity) AS qt_revertidos FROM reversas_universo GROUP BY 1),
reversas_por_dia AS (
  SELECT p.data_compra AS dia, SUM(r.qt_revertidos) AS reversas
  FROM reversas_por_pedido r JOIN todos_pedidos p USING (order_name)
  WHERE p.data_compra >= '2025-01-01' GROUP BY 1)
SELECT COALESCE(v.dia, r.dia) AS dia, COALESCE(v.vendas,0) AS vendas, COALESCE(r.reversas,0) AS reversas
FROM vendas_por_dia v FULL OUTER JOIN reversas_por_dia r USING (dia)
ORDER BY dia
"""
df_cohort = client.query(query_cohort).to_dataframe(create_bqstorage_client=False)
df_cohort["dia"] = pd.to_datetime(df_cohort["dia"])
HOJE_D = pd.Timestamp.today().normalize()
print(f"{len(df_cohort)} dias | vendas {df_cohort['vendas'].sum():,.0f} | reversas {df_cohort['reversas'].sum():,.0f}")
df_cohort.tail()

582 dias | vendas 4,950,619 | reversas 369,652


,dia,vendas,reversas
577,2026-08-01,7310,15.00
578,2026-08-02,9202,27.00
579,2026-08-03,8315,11.00
580,2026-08-04,8922,0.00
581,2026-08-05,272,0.00


In [48]:
# --- SoT em SQL: as DUAS queries oficiais do OKR de T&D (mensal + semanal) ---
# Fonte única da verdade = os arquivos .sql versionados. O texto do SQL NÃO é duplicado aqui:
# notebook, auditoria e analista externo têm de rodar exatamente o mesmo código.
#   sql/td_cohort_mensal.sql   -> KR do MBR   (inclui maduro_em / status)
#   sql/td_cohort_semanal.sql  -> HM semanal  (inclui reportavel_em / status / is_w2 / média móvel)
from pathlib import Path

# Caminho absoluto da pasta do notebook — não usar Path.cwd()/caminho relativo,
# pois o cwd do kernel pode não corresponder à pasta do notebook (ou até estar inválido).
SQL_DIR = Path("/Users/insider/LA_Coding_Projects/analyses/tempo_reversa_pos_entrega/sql")
if not SQL_DIR.exists():
    raise FileNotFoundError(
        f"Pasta sql/ não encontrada em {SQL_DIR}. Ajuste SQL_DIR para o caminho absoluto correto."
    )


def run_sql_file(nome):
    """Executa um .sql versionado e devolve o DataFrame."""
    sql = (SQL_DIR / nome).read_text(encoding="utf-8")
    return client.query(sql).to_dataframe(create_bqstorage_client=False)


df_kr_mensal = run_sql_file("td_cohort_mensal.sql")
df_hm_semanal = run_sql_file("td_cohort_semanal.sql")

print(f"mensal : {len(df_kr_mensal):>3} meses   "
      f"({df_kr_mensal['mes_referencia'].min()} -> {df_kr_mensal['mes_referencia'].max()})")
print(f"semanal: {len(df_hm_semanal):>3} semanas "
      f"({df_hm_semanal['semana_ini'].min()} -> {df_hm_semanal['semana_ini'].max()})")


mensal :   7 meses   (2026-01 -> 2026-07)
semanal:  84 semanas (2024-12-30 -> 2026-08-03)


## 12. KR mensal (MBR) — headline do "mês passado"

`T&D% = reversas do cohort do mês ÷ vendas do mês`. Headline = último mês cuja maturação (dia 15 de M+1) já passou; demais = `em maturação`.

In [ ]:
# KR mensal (MBR) — lido DIRETO de sql/td_cohort_mensal.sql.
# Toda a premissa, inclusive a régua de maturação (dia 15 de M+1), vive na query.
# Aqui só se renomeia coluna, converte fração -> % e desenha.
kr = df_kr_mensal.rename(columns={
    "mes_referencia": "mes",
    "qt_itens_vendidos": "vendas",
    "qt_itens_revertidos": "reversas",
}).copy()
kr["td_pct"] = (kr["pct_reversas_sobre_vendas"] * 100).round(2)
kr["status"] = kr["status"].replace({"em maturacao": "em maturação"})
kr = kr.sort_values("mes").reset_index(drop=True)
display(kr[["mes", "vendas", "reversas", "td_pct", "status", "maduro_em"]].tail(12))

oficiais = kr[kr["status"].str.startswith("oficial")]
head = oficiais.iloc[-1]
prox = kr[kr["status"] == "em maturação"].iloc[0] if (kr["status"] == "em maturação").any() else None
print(f"KR VIGENTE (último mês maduro): {head['mes']} = {head['td_pct']:.2f}%")
if prox is not None:
    print(f"Próxima leitura: {prox['mes']} (provisório {prox['td_pct']:.2f}%) — oficial em {prox['maduro_em']}")

fig = px.bar(oficiais.tail(12), x="mes", y="td_pct", text="td_pct",
             title=f"KR mensal (SoT cohort · SQL) — T&D% oficial · headline {head['mes']} = {head['td_pct']:.2f}%",
             labels={"mes": "", "td_pct": "T&D (%)"}, color_discrete_sequence=["#2b6cb0"])
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.update_layout(height=420)
fig.show()


,mes,vendas,reversas,td_pct,status,maduro_em
0,2026-01,217787,"18,329.00",8.42,oficial (maduro),2026-02-15
1,2026-02,228529,"19,411.00",8.49,oficial (maduro),2026-03-15
2,2026-03,376554,"35,137.00",9.33,oficial (maduro),2026-04-15
3,2026-04,244750,"18,305.00",7.48,oficial (maduro),2026-05-15
4,2026-05,195610,"14,800.00",7.57,oficial (maduro),2026-06-15
5,2026-06,185498,"13,956.00",7.52,oficial (maduro),2026-07-15
6,2026-07,188788,"12,099.00",6.41,em maturação,2026-08-15


KR VIGENTE (último mês maduro): 2026-06 = 7.52%
Próxima leitura: 2026-07 (provisório 6.41%) — oficial em 2026-08-15


## 13. HM semanal contínuo (cohort · W-2)

Série semanal por **semana de compra** (cohort lifetime). Reportável até **W-2 = [D-21, D-15]**; semanas mais recentes ainda maturando (subestimadas).

In [ ]:
# HM semanal — lido DIRETO de sql/td_cohort_semanal.sql.
# Semana = segunda a domingo (DATE_TRUNC(d, WEEK(MONDAY))); régua W-2 e média móvel 4 sem vêm da query.
wk = df_hm_semanal.rename(columns={
    "qt_itens_vendidos": "vendas",
    "qt_itens_revertidos": "reversas",
}).copy()
wk["semana_ini"] = pd.to_datetime(wk["semana_ini"])
wk["td_pct"] = (wk["pct_reversas_sobre_vendas"] * 100).round(2)
wk["media_movel_4s"] = (wk["media_movel_4s"] * 100).round(2)
wk["reportavel"] = wk["status"].eq("reportavel (<=W-2)")
wk = wk.sort_values("semana_ini").reset_index(drop=True)
display(wk[["semana_ini", "vendas", "reversas", "td_pct", "media_movel_4s", "reportavel"]].tail(8))

v = wk[wk["is_w2"]].iloc[0]
print(f"HM W-2 [{v['semana_ini'].date()}..{v['semana_fim']}] = {v['td_pct']:.2f}%"
      f"  (média móvel 4 sem: {v['media_movel_4s']:.2f}%)")

rep = wk[wk["reportavel"]]; imat = wk[~wk["reportavel"]]
fig = go.Figure()
if len(imat):
    fig.add_vrect(x0=imat["semana_ini"].min(), x1=wk["semana_ini"].max(), fillcolor="#f6ad55", opacity=0.12, line_width=0,
                  annotation_text="imaturo (> W-2)", annotation_position="top left")
fig.add_trace(go.Scatter(x=rep["semana_ini"], y=rep["td_pct"], mode="lines+markers",
                         name="T&D cohort semanal (≤ W-2)", line=dict(color="#2b6cb0")))
if len(imat):
    link = pd.concat([rep.tail(1), imat])
    fig.add_trace(go.Scatter(x=link["semana_ini"], y=link["td_pct"], mode="lines+markers",
                             name="ainda maturando (> W-2)", line=dict(color="#a0aec0", dash="dash")))
fig.add_trace(go.Scatter(x=rep["semana_ini"], y=rep["media_movel_4s"], mode="lines",
                         name="média móvel 4 sem", line=dict(color="#dd6b20", width=3)))
fig.add_vline(x=v["semana_ini"], line=dict(color="#38a169", dash="dot"))
fig.add_annotation(x=v["semana_ini"], y=v["td_pct"], text=f"W-2 (HM) {v['td_pct']:.2f}%",
                   showarrow=True, arrowhead=2, arrowcolor="#38a169", font=dict(color="#276749"))
fig.update_layout(title="T&D COHORT semanal (%) — reversas das compras da semana ÷ vendas da semana (SoT · SQL)",
                  xaxis_title="semana de compra (início)", yaxis_title="T&D (%)", height=470)
fig.show()


,semana_ini,vendas,reversas,td_pct,media_movel_4s,reportavel
76,2026-06-15,35005,"2,884.00",8.24,7.38,True
77,2026-06-22,31972,"2,739.00",8.57,7.70,True
78,2026-06-29,43676,"3,632.00",8.32,8.15,True
79,2026-07-06,43970,"3,470.00",7.89,8.25,True
80,2026-07-13,37707,"2,880.00",7.64,8.10,True
81,2026-07-20,38371,"2,309.00",6.02,7.47,False
82,2026-07-27,53187,768.00,1.44,5.75,False
83,2026-08-03,17509,11.00,0.06,3.79,False


HM W-2 [2026-07-13..2026-07-19] = 7.64%  (média móvel 4 sem: 8.10%)


## 14. Export dos CSVs


In [ ]:
import os
os.makedirs(OUT_DIR, exist_ok=True)

# --- Parte A: entrega -> reversa ---
df.to_csv(f"{OUT_DIR}/tempo_reversa_base_{DATA_TAG}.csv", index=False)
stats_tipo.to_csv(f"{OUT_DIR}/tempo_reversa_por_tipo_{DATA_TAG}.csv")
tab_bucket.to_csv(f"{OUT_DIR}/tempo_reversa_buckets_{DATA_TAG}.csv")
coorte.to_csv(f"{OUT_DIR}/tempo_reversa_coorte_mensal_{DATA_TAG}.csv")

# --- Parte B: compra -> reversa ---
df_compra.to_csv(f"{OUT_DIR}/tempo_compra_reversa_base_{DATA_TAG}.csv", index=False)
stats_tipo_compra.to_csv(f"{OUT_DIR}/tempo_compra_reversa_por_tipo_{DATA_TAG}.csv")
tab_bucket_compra.to_csv(f"{OUT_DIR}/tempo_compra_reversa_buckets_{DATA_TAG}.csv")
coorte_compra.to_csv(f"{OUT_DIR}/tempo_compra_reversa_coorte_mensal_{DATA_TAG}.csv")
tab_dec.to_csv(f"{OUT_DIR}/tempo_decomposicao_compra_entrega_reversa_{DATA_TAG}.csv", index=False)

# --- Parte C: OKR T&D (SoT cohort) ---
kr[["mes","vendas","reversas","td_pct","status"]].to_csv(f"{OUT_DIR}/okr_td_kr_mensal_{DATA_TAG}.csv", index=False)
wk.to_csv(f"{OUT_DIR}/okr_td_hm_semanal_{DATA_TAG}.csv", index=False)
df_cohort.to_csv(f"{OUT_DIR}/okr_td_cohort_diario_{DATA_TAG}.csv", index=False)
print("  Parte C: okr_td_{kr_mensal,hm_semanal,cohort_diario}")


  Parte C: okr_td_{kr_mensal,hm_semanal,cohort_diario}


## 15. Sumário executivo

*Execução: 2026-07-27 · entregas dos últimos 12 meses · pedidos válidos (filtros T&D) · fonte da reversa = Troquecommerce (`created_at`).*
*Base: 113.389 reversas casadas com entrega; 3,7% abertas antes da `delivered_date` (ruído) excluídas → 109.232 válidas.*

**1. A decisão de reverter é rápida e concentrada nas primeiras 48h.**
Mediana de **2,1 dias** entre a entrega e a abertura da reversa. **48,6%** das reversas são abertas em até 2 dias, **78,8%** em até 7 dias e **91,2%** em até 15 dias. A janela de intervenção (retenção/UX) é curtíssima.

**2. Troca domina o volume — ~8× mais que devolução.**
Troca = **96.174** reversas · Devolução = **11.514** (via Troquecommerce). A Insider canaliza a insatisfação para troca, não para saída de caixa via devolução.

**3. Devolução decide mais rápido e com cauda mais curta que troca.**
Devolução: mediana 2,0d, **p90 9,0d**, 86% em ≤7d. Troca: mediana 2,1d, **p90 14,2d**, 78% em ≤7d. Quem vai devolver bate o martelo cedo; a troca tem mais gente que "pensa mais" (experimenta, avalia grade/tamanho) e volta em 1–2 semanas.

**4. Distribuição por bucket:** 0–2d **48,6%** · 3–7d **30,2%** · 8–15d **12,4%** · 16–30d **5,7%** · 31d+ **3,1%**.

### Implicações e próximos passos
- **Operação/UX:** ação de retenção tem valor máximo nas **primeiras 48h pós-entrega** (grade alternativa, ajuste de tamanho, prova virtual). Depois de 7 dias, ~79% da reversa já foi disparada.
- **Aprofundar causa (moonshot):** cruzar `dias_ate_reversa` × `return_reason` × categoria para separar reversa por **fit/tamanho** (decisão rápida, 0–2d) de **defeito/qualidade** (tende a aparecer mais tarde) — muda a alavanca (curadoria de grade vs. QA de fornecedor).
- **Rigor estatístico:** os meses de entrega mais recentes têm **censura à direita**; para tendência limpa, reprocessar excluindo os últimos ~30 dias de entrega ou usar coorte fechada.


---

### Parte B — Compra → reversa (fecha o ciclo)

- **Mediana de 9 dias** entre a compra e a abertura da reversa (p25 6d · p90 23d). **0 reversas** ocorrem antes da compra (métrica limpa, sem exclusões).
- **O gargalo é o frete, não a indecisão:** decompondo, a mediana **compra→entrega (envio) é ~5 dias** e **entrega→reversa (decisão) é ~2 dias**. O cliente decide rápido; o relógio corre no transporte.
- **Alavanca dupla:** reduzir lead time de entrega antecipa (e pode reduzir) reversas por ansiedade/atraso; agir na experiência nas primeiras 48h pós-entrega ataca a janela de decisão. As duas frentes são complementares.

---

### Parte C — OKR de T&D (Source of Truth · COHORT) — execução 2026-08-05

- **Definição:** por **coorte de compra** — reversa atribuída ao mês/semana da **compra** do pedido (via `order_name`), numerador **lifetime**. `T&D% = reversas do cohort ÷ vendas do cohort`. Materializada em [`sql/td_cohort_mensal.sql`](sql/td_cohort_mensal.sql) e [`sql/td_cohort_semanal.sql`](sql/td_cohort_semanal.sql).
- **KR mensal (headline = último mês maduro):** **jun/26 = 7,52%** (vigente). jul/26 (6,41% provisório, imaturo) fica oficial em **15/ago**. Pico do ano: **mar/26 = 9,33%**.
- **HM semanal (W-2 = semana de 13/jul):** **7,64%** (média móvel 4 sem: 8,10%). Semanas > W-2 ainda maturando — a semana de 27/jul marca 1,44% por **censura à direita**, não por melhora.
- **Reconciliação vs. legado (7 meses, jan–jul/26):** denominador bate em **zero** nos 7 meses → a **ancoragem de data é a única** diferença entre os métodos. Ponte de numerador fecha 100%: legado 142.768 = interseção 130.503 + 12.265 (reversa em 2026 de compra anterior); cohort 132.037 = 130.503 + 1.534 (cauda já aberta em ago/26).
- **Achado material:** **44% do numerador legado de jan/26** (10.151 de 23.048) vem de compras de **2025** — Black Friday chegando à janela de troca em janeiro, dividida pelas vendas de janeiro. Legado lê **10,58%**, cohort lê **8,42%**. O pico de janeiro no legado é **artefato de mistura de safras**; e um mês de vendas alto **reduz** mecanicamente o T&D legado do próprio mês. Isso é fatal para um KR — e é a razão de adotar o cohort.
- **Dívida aberta:** `return_quantity` está dentro do `DISTINCT` nos **dois** métodos → exposição a dupla contagem (simétrica hoje, frágil sempre). Ver [`AUDITORIA_PREMISSAS_OKR_TD.md`](AUDITORIA_PREMISSAS_OKR_TD.md) §6.1.


# Parte D — Auditoria de metodologia: coorte (oficial) vs. mensal (legado)

Duas definições de T&D convivem na empresa e produzem números diferentes. Esta parte mantém **as duas** lado a lado e mede exatamente de onde vem a divergência.

**Revisão de 2026-08-04:** o universo de reversas contado pelas duas definições foi harmonizado — as duas somam **exatamente o mesmo conjunto de reversas não canceladas** (mesmo dedup `order_name,id_reversa,sku`, sem exigir pedido `paid`/loja elegível nem SKU casado no numerador). A única diferença estrutural remanescente é a **ancoragem de data**.

| | **Oficial — coorte de compra** (Parte C, SoT) | **Legado — mensal por fluxo** (auditoria) |
|---|---|---|
| Numerador (universo de reversa) | **idêntico** ao legado — todas as reversas não canceladas | **idêntico** ao coorte |
| Ancoragem do numerador | mês/semana em que o pedido foi **comprado** | mês em que a reversa foi **criada** |
| Denominador | itens comprados no mês (`paid`/filtros de cupom/lojas) | itens comprados no mês (idêntico) |
| Chave de vínculo p/ ancorar | `order_name` (ordem, não mais SKU) | `order_name` |
| Responde | qualidade do lote vendido no mês | carga operacional de reversa no mês |

**Casos de auditoria:** `Tech T-shirt Gola U` e `The Perfect Top`, últimos 4 meses fechados.

**Definição de produto:** `family` em `insider-data-lake.integrated.skus` (dedup pelo `ingestion_date` mais recente). Escopo **núcleo**: exclui outlet (`Defeitos Leves/Moderados/Graves`), kits, co-branded/uniformes B2B e extensões de linha (`Cropped`, `Halter`, `V`, `Asymmetric`, `T-shirt`). Validado: cobre 100% dos SKUs de reversa do período.

### Limitações declaradas
- **Censura à direita:** a coorte do último mês ainda está em maturação (D.11 estima o quanto falta). Parte do delta entre métodos no mês corrente é artefato de janela, não de metodologia.
- **Método legado não é atribuível a coorte:** não serve para avaliar qualidade de um lote de vendas.
- **Escopo núcleo** não reconcilia com o total da companhia por construção.
- **Trade-off aceito na harmonização:** ao ancorar por `order_name` (não mais `order_name`+`sku`), uma reversa é atribuída ao mês de compra do pedido mesmo que o SKU revertido não conste como item efetivamente comprado naquele pedido (situação residual, ver `item_no_pedido` em D.4) — mesmo comportamento que a definição legada sempre teve.

## D.1 Parâmetros da auditoria

In [ ]:
PRODUTOS_AUDITORIA = ["Tech T-shirt Gola U", "The Perfect Top"]
N_MESES_AUDITORIA = 4

MES_FIM_AUD = HOJE.to_period("M") - 1                      # último mês fechado
MES_INI_AUD = MES_FIM_AUD - (N_MESES_AUDITORIA - 1)
D_INI_AUD = MES_INI_AUD.start_time.date().isoformat()
D_FIM_AUD = MES_FIM_AUD.end_time.date().isoformat()
MESES_AUD = [str(MES_INI_AUD + i) for i in range(N_MESES_AUDITORIA)]
FAM_SQL = ", ".join(f"'{p}'" for p in PRODUTOS_AUDITORIA)

# filtro de pedido válido — idêntico às Partes A/B/C, replicado literalmente p/ garantir comparabilidade
FILTRO_PEDIDO = """
    order_status = 'paid' AND is_cancelled = FALSE
    AND (coupon_code IS NULL OR (
      NOT STARTS_WITH(coupon_code, 'TF-')  AND NOT STARTS_WITH(coupon_code, 'TFIN')
      AND NOT STARTS_WITH(coupon_code, 'IR') AND NOT coupon_code LIKE '%Item errado%'))
    AND order_name IS NOT NULL AND processed_at IS NOT NULL
    AND store IN ('shopify_insider-world', 'shopify_insider-store-loja')
"""

CTE_SKUS = f"""
skus_dedup AS (
  SELECT sku, family, product_name, gender, color, size
  FROM `insider-data-lake.integrated.skus`
  QUALIFY ROW_NUMBER() OVER (PARTITION BY sku ORDER BY ingestion_date DESC) = 1
),
skus AS (SELECT * FROM skus_dedup WHERE family IN ({FAM_SQL}))
"""

print(f"Produtos : {PRODUTOS_AUDITORIA}")
print(f"Janela   : {D_INI_AUD} a {D_FIM_AUD}  ({', '.join(MESES_AUD)})")

Produtos : ['Tech T-shirt Gola U', 'The Perfect Top']
Janela   : 2026-04-01 a 2026-07-31  (2026-04, 2026-05, 2026-06, 2026-07)


## D.2 Allowlist de SKU e QA da fonte

A lista de SKUs é impressa para revisão manual — é a definição operacional de "produto" nesta auditoria.

In [ ]:
q_skus_aud = f"WITH {CTE_SKUS} SELECT sku, family, product_name, gender, color, size FROM skus ORDER BY family, sku"
df_skus_aud = client.query(q_skus_aud).to_dataframe(create_bqstorage_client=False)

print("SKUs por família e gênero:")
display(df_skus_aud.groupby(["family", "gender"]).size().rename("n_sku").to_frame())
print(f"total de SKUs na allowlist: {len(df_skus_aud)}")
display(df_skus_aud.head(10))

q_qa_aud = f"""
WITH {CTE_SKUS},
rev AS (
  SELECT DISTINCT order_name, id_reversa, sku, created_at, return_quantity
  FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br`
  WHERE status <> 'Cancelado' AND order_name IS NOT NULL AND sku IS NOT NULL
    AND id_reversa IS NOT NULL AND created_at IS NOT NULL
    AND DATE(created_at, 'America/Sao_Paulo') BETWEEN '{D_INI_AUD}' AND '{D_FIM_AUD}'
)
SELECT
  (SELECT COUNT(*) FROM rev) AS n_linhas_dedup,
  (SELECT COUNT(*) FROM (SELECT DISTINCT order_name, id_reversa, sku FROM rev)) AS n_chaves,
  (SELECT COUNTIF(s.sku IS NULL) FROM (SELECT DISTINCT sku FROM rev) r LEFT JOIN skus_dedup s USING (sku)) AS skus_sem_dicionario,
  (SELECT COUNT(*) FROM rev r LEFT JOIN skus_dedup s USING (sku) WHERE s.sku IS NULL) AS linhas_sem_dicionario
"""
qa_aud = client.query(q_qa_aud).to_dataframe(create_bqstorage_client=False).iloc[0]
dup_aud = int(qa_aud.n_linhas_dedup - qa_aud.n_chaves)
cob_aud = 1 - qa_aud.linhas_sem_dicionario / qa_aud.n_linhas_dedup

print("\n--- QA da fonte de reversa (janela da auditoria) ---")
print(f"linhas dedup (order_name,id_reversa,sku,created_at,return_quantity): {qa_aud.n_linhas_dedup:,}")
print(f"chaves distintas (order_name,id_reversa,sku)                      : {qa_aud.n_chaves:,}")
print(f"linhas excedentes por chave (risco de dupla contagem)             : {dup_aud:,}")
print(f"cobertura de integrated.skus sobre os SKUs de reversa             : {cob_aud:.2%}"
      f"  ({qa_aud.skus_sem_dicionario} SKUs / {qa_aud.linhas_sem_dicionario} linhas sem dicionário)")
if cob_aud < 1:
    print("ATENÇÃO: cobertura < 100% — revisar o dicionário antes de usar os números por produto.")

SKUs por família e gênero:


n_sku
family              gender       
Tech T-shirt Gola U female    202
                    male      230
The Perfect Top     female    137

total de SKUs na allowlist: 569


,sku,family,product_name,gender,color,size
0,102010100103,Tech T-shirt Gola U,Tech T-shirt Gola U Masculino,male,Preto,PP
1,102010100104,Tech T-shirt Gola U,Tech T-shirt Gola U Masculino,male,Preto,P
2,102010100105,Tech T-shirt Gola U,Tech T-shirt Gola U Masculino,male,Preto,M
3,102010100106,Tech T-shirt Gola U,Tech T-shirt Gola U Masculino,male,Preto,G
4,102010100107,Tech T-shirt Gola U,Tech T-shirt Gola U Masculino,male,Preto,GG
5,102010100108,Tech T-shirt Gola U,Tech T-shirt Gola U Masculino,male,Preto,XGG
6,102010100119,Tech T-shirt Gola U,Tech T-shirt Gola U Masculino,male,Preto,XXGG
7,102010100203,Tech T-shirt Gola U,Tech T-shirt Gola U Masculino,male,Branco,PP
8,102010100204,Tech T-shirt Gola U,Tech T-shirt Gola U Masculino,male,Branco,P
9,102010100205,Tech T-shirt Gola U,Tech T-shirt Gola U Masculino,male,Branco,M



--- QA da fonte de reversa (janela da auditoria) ---
linhas dedup (order_name,id_reversa,sku,created_at,return_quantity): 63,327
chaves distintas (order_name,id_reversa,sku)                      : 63,326
linhas excedentes por chave (risco de dupla contagem)             : 1
cobertura de integrated.skus sobre os SKUs de reversa             : 100.00%  (0 SKUs / 0 linhas sem dicionário)


## D.3 Método oficial — coorte por produto (harmonizado com o universo legado)

Numerador: **todas** as reversas não canceladas dos SKUs do produto (sem exigir pedido válido nem SKU casado ao item comprado — mesmo universo da Parte D.6/legado), ancoradas ao mês de compra via `order_name` contra **qualquer pedido** (`todos_pedidos`, sem os filtros `paid`/loja/cupom). Denominador: itens comprados no mês, inalterado (`paid`/filtros de cupom/lojas).

In [ ]:
q_cohort_prod = f"""
WITH {CTE_SKUS},
compras_filtradas AS (
  SELECT DISTINCT order_id, order_name, DATE(TIMESTAMP(processed_at),'America/Sao_Paulo') AS data_compra
  FROM `insider-data-lake.business.insider_orders`
  WHERE {FILTRO_PEDIDO}
),
itens_comprados AS (
  SELECT s.family, c.data_compra, i.sku, SUM(i.quantity) AS qt_comprados
  FROM compras_filtradas c
  JOIN `insider-data-lake.business.insider_order_items` i ON c.order_id = i.order_id
  JOIN skus s ON i.sku = s.sku
  WHERE c.data_compra BETWEEN '{D_INI_AUD}' AND '{D_FIM_AUD}'
  GROUP BY 1,2,3
),
vendas_por_dia AS (
  SELECT family, data_compra AS dia, SUM(qt_comprados) AS vendas FROM itens_comprados GROUP BY 1,2
),
-- data de compra de QUALQUER pedido — mesmo universo de pedido que a definição legada aceita
-- implicitamente (ela nunca valida o pedido da reversa). Só ancora a data.
todos_pedidos AS (
  SELECT order_name, MIN(DATE(TIMESTAMP(processed_at),'America/Sao_Paulo')) AS data_compra
  FROM `insider-data-lake.business.insider_orders`
  WHERE order_name IS NOT NULL AND processed_at IS NOT NULL
  GROUP BY 1
),
-- universo de reversa IDÊNTICO ao da definição legada: sem exigir pedido válido nem SKU casado.
reversas_universo AS (
  SELECT DISTINCT order_name, id_reversa, sku, SAFE_CAST(return_quantity AS FLOAT64) AS return_quantity
  FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br`
  WHERE status <> 'Cancelado' AND order_name IS NOT NULL AND sku IS NOT NULL AND id_reversa IS NOT NULL
    AND created_at IS NOT NULL
),
reversas_produto AS (
  SELECT s.family, ru.order_name, SUM(ru.return_quantity) AS qt_revertidos
  FROM reversas_universo ru JOIN skus s ON ru.sku = s.sku
  GROUP BY 1,2
),
reversas_por_dia AS (
  SELECT rp.family, tp.data_compra AS dia, SUM(rp.qt_revertidos) AS reversas
  FROM reversas_produto rp JOIN todos_pedidos tp USING (order_name)
  WHERE tp.data_compra BETWEEN '{D_INI_AUD}' AND '{D_FIM_AUD}'
  GROUP BY 1,2
)
SELECT COALESCE(v.family, r.family) AS family, COALESCE(v.dia, r.dia) AS dia,
       COALESCE(v.vendas, 0) AS vendas, COALESCE(r.reversas, 0) AS reversas
FROM vendas_por_dia v FULL OUTER JOIN reversas_por_dia r USING (family, dia)
ORDER BY 1, 2
"""
df_cohort_prod = client.query(q_cohort_prod).to_dataframe(create_bqstorage_client=False)
df_cohort_prod["dia"] = pd.to_datetime(df_cohort_prod["dia"])
print(f"{len(df_cohort_prod)} linhas (produto x dia) | vendas {df_cohort_prod['vendas'].sum():,.0f}"
      f" | reversas {df_cohort_prod['reversas'].sum():,.0f}")
df_cohort_prod.head()

244 linhas (produto x dia) | vendas 208,559 | reversas 12,751


,family,dia,vendas,reversas
0,Tech T-shirt Gola U,2026-04-01,1063,71.00
1,Tech T-shirt Gola U,2026-04-02,1035,65.00
2,Tech T-shirt Gola U,2026-04-03,1259,74.00
3,Tech T-shirt Gola U,2026-04-04,1187,71.00
4,Tech T-shirt Gola U,2026-04-05,1273,62.00


## D.4 Detalhe das reversas com as três datas

Grão: uma linha por `(order_name, id_reversa, sku)`. Universo: SKU na allowlist **e** (reversa aberta na janela **ou** compra na janela) — o `OR` é necessário porque o coorte precisa de reversas fora da janela e o método mensal precisa de compras fora dela.

`data_compra` vem de **qualquer pedido** (`todos_pedidos`, sem exigir `paid`/loja/cupom) — mesmo universo de pedido que a definição legada aceita implicitamente. `item_no_pedido` é só diagnóstico: informa se o SKU revertido de fato consta como item comprado naquele pedido (independente da validade do pedido).

In [ ]:
q_rev_aud = f"""
WITH {CTE_SKUS},
todos_pedidos AS (
  SELECT order_name, MIN(DATE(TIMESTAMP(processed_at),'America/Sao_Paulo')) AS data_compra
  FROM `insider-data-lake.business.insider_orders`
  WHERE order_name IS NOT NULL AND processed_at IS NOT NULL
  GROUP BY 1
),
entrega AS (
  SELECT order_name, MIN(delivered_date) AS delivered_date
  FROM `insider-lake-sensitive.integrated_br.shippings_br`
  WHERE delivered_date IS NOT NULL GROUP BY 1
),
rev AS (
  SELECT DISTINCT order_name, id_reversa, sku, created_at, reverse_type,
         SAFE_CAST(return_quantity AS FLOAT64) AS return_quantity
  FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br`
  WHERE status <> 'Cancelado' AND order_name IS NOT NULL AND sku IS NOT NULL
    AND id_reversa IS NOT NULL AND created_at IS NOT NULL
),
itens_pedido AS (
  SELECT DISTINCT o.order_name, i.sku
  FROM `insider-data-lake.business.insider_orders` o
  JOIN `insider-data-lake.business.insider_order_items` i ON o.order_id = i.order_id
  WHERE i.sku IS NOT NULL AND o.order_name IS NOT NULL
)
SELECT
  s.family, r.order_name, r.id_reversa, r.sku, r.reverse_type, r.return_quantity,
  DATE(r.created_at, 'America/Sao_Paulo') AS data_reversa,
  tp.data_compra,
  DATE(e.delivered_date) AS data_entrega,
  ip.sku IS NOT NULL AS item_no_pedido
FROM rev r
JOIN skus s ON r.sku = s.sku
LEFT JOIN todos_pedidos tp USING (order_name)
LEFT JOIN entrega e USING (order_name)
LEFT JOIN itens_pedido ip ON r.order_name = ip.order_name AND r.sku = ip.sku
WHERE DATE(r.created_at, 'America/Sao_Paulo') BETWEEN '{D_INI_AUD}' AND '{D_FIM_AUD}'
   OR tp.data_compra BETWEEN '{D_INI_AUD}' AND '{D_FIM_AUD}'
"""
df_rev_aud = client.query(q_rev_aud).to_dataframe(create_bqstorage_client=False)

for _c in ["data_reversa", "data_compra", "data_entrega"]:
    df_rev_aud[_c] = pd.to_datetime(df_rev_aud[_c])
df_rev_aud["mes_reversa"] = df_rev_aud["data_reversa"].dt.to_period("M").astype(str)
df_rev_aud["mes_compra"] = df_rev_aud["data_compra"].dt.to_period("M").astype(str)
df_rev_aud["dias_compra_reversa"] = (df_rev_aud["data_reversa"] - df_rev_aud["data_compra"]).dt.days
df_rev_aud["dias_entrega_reversa"] = (df_rev_aud["data_reversa"] - df_rev_aud["data_entrega"]).dt.days
df_rev_aud["lag_meses"] = (df_rev_aud["data_reversa"].dt.to_period("M").astype("int64")
                           - df_rev_aud["data_compra"].dt.to_period("M").astype("int64"))
df_rev_aud.loc[df_rev_aud["data_compra"].isna(), "lag_meses"] = np.nan
df_rev_aud["tipo"] = df_rev_aud["reverse_type"].map(mapa_tipo).fillna("Outros")

print(f"linhas de reversa            : {len(df_rev_aud):,}")
print(f"sem pedido casado (qualquer status): {df_rev_aud['data_compra'].isna().sum():,}"
      f"  ({df_rev_aud['data_compra'].isna().mean():.2%})")
print(f"SKU não consta no pedido     : {(~df_rev_aud['item_no_pedido']).sum():,}")
print(f"sem entrega casada           : {df_rev_aud['data_entrega'].isna().sum():,}")
df_rev_aud.head()

linhas de reversa            : 14,253
sem pedido casado (qualquer status): 0  (0.00%)
SKU não consta no pedido     : 0
sem entrega casada           : 65


,family,order_name,id_reversa,sku,reverse_type,return_quantity,data_reversa,data_compra,data_entrega,item_no_pedido,mes_reversa,mes_compra,dias_compra_reversa,dias_entrega_reversa,lag_meses,tipo
0,Tech T-shirt Gola U,IN-3902984,2ff0b5a8-fc1b-4c6b-9978-c0a90aede4fa,102010109307,Troca,2.00,2026-04-16,2026-04-15,2026-04-16,True,2026-04,2026-04,1,0.00,0.00,Troca
1,Tech T-shirt Gola U,IN-3943320,99733155-f809-446b-9bed-9df34d6d3246,202010102805,Devolução,1.00,2026-05-12,2026-04-29,2026-05-06,True,2026-05,2026-04,13,6.00,1.00,Devolução
2,Tech T-shirt Gola U,IN-4103632,b46fcb2a-fbaa-42f8-b465-a3bd2478f5ac,1094010110307,Troca,1.00,2026-07-09,2026-06-26,2026-07-06,True,2026-07,2026-06,13,3.00,1.00,Troca
3,Tech T-shirt Gola U,IN-3881266,1d3bd450-c38f-4bda-8898-3ec3234c665f,2094010118404,Devolução,1.00,2026-04-17,2026-04-09,2026-04-15,True,2026-04,2026-04,8,2.00,0.00,Devolução
4,Tech T-shirt Gola U,IN-4056253,0592e134-7b68-49cf-aa3d-ae125193357c,2094010114605,Troca,1.00,2026-06-11,2026-06-07,2026-06-11,True,2026-06,2026-06,4,0.00,0.00,Troca


## D.5–D.7 Os dois métodos lado a lado

Denominador **idêntico** e numerador do **mesmo universo de reversa** nos dois métodos — a única diferença agora é a ancoragem de data (compra vs. criação da reversa).

In [ ]:
# --- D.5 oficial (coorte) ---
_c = df_cohort_prod.copy()
_c["mes"] = _c["dia"].dt.to_period("M").astype(str)
cohort_prod = (_c[_c["mes"].isin(MESES_AUD)].groupby(["family", "mes"])
               .agg(vendas=("vendas", "sum"), reversas_cohort=("reversas", "sum")).reset_index())
cohort_prod["td_pct_cohort"] = (cohort_prod["reversas_cohort"] / cohort_prod["vendas"] * 100).round(2)

# --- D.6 legado (mensal) ---
mensal_prod = (df_rev_aud[df_rev_aud["mes_reversa"].isin(MESES_AUD)]
               .groupby(["family", "mes_reversa"])["return_quantity"].sum()
               .rename("reversas_criadas").reset_index().rename(columns={"mes_reversa": "mes"}))

# --- D.7 lado a lado ---
aud_lado = cohort_prod.merge(mensal_prod, on=["family", "mes"], how="left")
aud_lado["td_pct_mensal"] = (aud_lado["reversas_criadas"] / aud_lado["vendas"] * 100).round(2)
aud_lado["delta_pp"] = (aud_lado["td_pct_mensal"] - aud_lado["td_pct_cohort"]).round(2)
aud_lado["delta_rel_pct"] = ((aud_lado["td_pct_mensal"] / aud_lado["td_pct_cohort"] - 1) * 100).round(1)
aud_lado["maduro_em"] = (pd.PeriodIndex(aud_lado["mes"], freq="M").to_timestamp()
                         + pd.offsets.MonthBegin(1) + pd.Timedelta(days=14))
aud_lado["status_cohort"] = np.where(HOJE >= aud_lado["maduro_em"], "oficial (maduro)", "em maturação")

display(aud_lado[["family", "mes", "vendas", "reversas_cohort", "reversas_criadas", "td_pct_cohort",
                  "td_pct_mensal", "delta_pp", "delta_rel_pct", "status_cohort"]])

aud_consol = aud_lado.groupby("mes").agg(vendas=("vendas", "sum"),
                                         reversas_cohort=("reversas_cohort", "sum"),
                                         reversas_criadas=("reversas_criadas", "sum")).reset_index()
aud_consol["td_pct_cohort"] = (aud_consol["reversas_cohort"] / aud_consol["vendas"] * 100).round(2)
aud_consol["td_pct_mensal"] = (aud_consol["reversas_criadas"] / aud_consol["vendas"] * 100).round(2)
aud_consol["delta_pp"] = (aud_consol["td_pct_mensal"] - aud_consol["td_pct_cohort"]).round(2)
print("Consolidado dos dois produtos:")
display(aud_consol)

assert aud_lado["vendas"].notna().all(), "denominador ausente em algum produto/mês"

,family,mes,vendas,reversas_cohort,reversas_criadas,td_pct_cohort,td_pct_mensal,delta_pp,delta_rel_pct,status_cohort
0,Tech T-shirt Gola U,2026-04,37258,"2,074.00","2,582.00",5.57,6.93,1.36,24.40,oficial (maduro)
1,Tech T-shirt Gola U,2026-05,28231,"1,569.00","1,717.00",5.56,6.08,0.52,9.40,oficial (maduro)
2,Tech T-shirt Gola U,2026-06,25084,"1,458.00","1,658.00",5.81,6.61,0.80,13.80,oficial (maduro)
3,Tech T-shirt Gola U,2026-07,30502,"1,510.00","1,626.00",4.95,5.33,0.38,7.70,em maturação
4,The Perfect Top,2026-04,31070,"2,629.00","3,012.00",8.46,9.69,1.23,14.50,oficial (maduro)
5,The Perfect Top,2026-05,22143,"1,504.00","1,789.00",6.79,8.08,1.29,19.00,oficial (maduro)
6,The Perfect Top,2026-06,14924,938.00,"1,149.00",6.29,7.70,1.41,22.40,oficial (maduro)
7,The Perfect Top,2026-07,19347,"1,069.00","1,218.00",5.53,6.30,0.77,13.90,em maturação


Consolidado dos dois produtos:


,mes,vendas,reversas_cohort,reversas_criadas,td_pct_cohort,td_pct_mensal,delta_pp
0,2026-04,68328,"4,703.00","5,594.00",6.88,8.19,1.31
1,2026-05,50374,"3,073.00","3,506.00",6.10,6.96,0.86
2,2026-06,40008,"2,396.00","2,807.00",5.99,7.02,1.03
3,2026-07,49849,"2,579.00","2,844.00",5.17,5.71,0.54


## D.8 Tempo entre compra e reversa, por coorte

Decomposição em **envio** (compra→entrega) e **decisão** (entrega→reversa), no padrão da Parte B. `dias < 0` excluídos das estatísticas de tempo e reportados.

In [ ]:
coh_det = df_rev_aud[(df_rev_aud["mes_compra"].isin(MESES_AUD)) & df_rev_aud["data_compra"].notna()].copy()
_neg = int((coh_det["dias_compra_reversa"] < 0).sum())
print(f"reversas do coorte: {len(coh_det):,} | dias_compra_reversa < 0 excluídos: {_neg}")
_cd = coh_det[coh_det["dias_compra_reversa"] >= 0]


def _stats_tempo(g, col):
    s = g[col].dropna()
    return pd.Series({"n": len(s), "mediana": s.median(), "media": s.mean(), "p90": s.quantile(0.90),
                      "pct_ate_7d": (s <= 7).mean() * 100, "pct_ate_15d": (s <= 15).mean() * 100,
                      "pct_ate_30d": (s <= 30).mean() * 100})


aud_tempo_compra = (_cd.groupby(["family", "mes_compra"])
                    .apply(_stats_tempo, "dias_compra_reversa", include_groups=False).round(2))
aud_tempo_entrega = (_cd[_cd["dias_entrega_reversa"] >= 0].groupby(["family", "mes_compra"])
                     .apply(_stats_tempo, "dias_entrega_reversa", include_groups=False).round(2))

print("\nCOMPRA -> reversa (total):")
display(aud_tempo_compra)
print("ENTREGA -> reversa (decisão do cliente):")
display(aud_tempo_entrega)

reversas do coorte: 11,972 | dias_compra_reversa < 0 excluídos: 0

COMPRA -> reversa (total):


n  mediana  media   p90  pct_ate_7d  \
family              mes_compra                                              
Tech T-shirt Gola U 2026-04    1,828.00     8.00  10.56 20.00       48.63   
                    2026-05    1,429.00     8.00  10.93 20.00       43.18   
                    2026-06    1,310.00     8.00   9.26 18.00       49.31   
                    2026-07    1,344.00     7.00   7.94 14.00       58.11   
The Perfect Top     2026-04    2,612.00     7.00   9.16 15.90       50.19   
                    2026-05    1,489.00     7.00   9.01 16.00       53.26   
                    2026-06      912.00     7.00   8.38 15.00       56.80   
                    2026-07    1,048.00     7.00   7.98 14.00       56.68   

                                pct_ate_15d  pct_ate_30d  
family              mes_compra                            
Tech T-shirt Gola U 2026-04           83.97        97.48  
                    2026-05           82.37        96.29  
                    2026-06           87.25        98.55  
                    2026-07           93.45       100.00  
The Perfect Top     2026-04           89.97        98.58  
                    2026-05           89.39        98.72  
                    2026-06           91.34        98.68  
                    2026-07           93.80        99.90

ENTREGA -> reversa (decisão do cliente):


n  mediana  media   p90  pct_ate_7d  \
family              mes_compra                                              
Tech T-shirt Gola U 2026-04    1,799.00     2.00   4.65 12.00       81.05   
                    2026-05    1,405.00     2.00   4.95 13.00       82.14   
                    2026-06    1,295.00     2.00   4.14 11.00       84.86   
                    2026-07    1,324.00     1.00   2.59  7.00       92.15   
The Perfect Top     2026-04    2,562.00     2.00   3.59  8.00       88.37   
                    2026-05    1,467.00     2.00   3.60  9.00       88.48   
                    2026-06      889.00     1.00   3.25  8.00       89.76   
                    2026-07    1,024.00     1.00   2.66  7.00       91.31   

                                pct_ate_15d  pct_ate_30d  
family              mes_compra                            
Tech T-shirt Gola U 2026-04           93.77        98.78  
                    2026-05           92.67        98.08  
                    2026-06           95.14        99.15  
                    2026-07           98.49       100.00  
The Perfect Top     2026-04           95.98        99.18  
                    2026-05           96.73        99.18  
                    2026-06           95.84        99.21  
                    2026-07           98.34       100.00

## D.9 Defasagem: reversa no mesmo mês da compra vs. transbordo

Para cada coorte de compra, em que mês a reversa foi aberta. `M+0` = mesmo mês da compra; `M+1` = mês seguinte. **É a causa mecânica da divergência entre os dois métodos.**

Coortes recentes aparecem com `M+0` artificialmente alto — ainda não tiveram tempo de gerar o `M+1` (ver D.11).

In [ ]:
_lg = _cd.copy()
_lg["lag_grp"] = np.where(_lg["lag_meses"] >= 3, "M+3+", "M+" + _lg["lag_meses"].astype(int).astype(str))
aud_lag_qt = _lg.pivot_table(index=["family", "mes_compra"], columns="lag_grp",
                             values="return_quantity", aggfunc="sum", fill_value=0)
_ordem_lag = [c for c in ["M+0", "M+1", "M+2", "M+3+"] if c in aud_lag_qt.columns]
aud_lag_qt = aud_lag_qt[_ordem_lag]
aud_lag_qt["total"] = aud_lag_qt.sum(axis=1)
aud_lag_pct = (aud_lag_qt[_ordem_lag].div(aud_lag_qt["total"], axis=0) * 100).round(1)

print("Reversas do coorte por mês de abertura (quantidade):")
display(aud_lag_qt)
print("Mesma tabela em % do coorte:")
display(aud_lag_pct)

# checagem de fechamento: total do lag == numerador do coorte
_chk = (aud_lag_qt["total"].reset_index().rename(columns={"mes_compra": "mes", "total": "lag_total"})
        .merge(cohort_prod[["family", "mes", "reversas_cohort"]], on=["family", "mes"]))
_chk["ok"] = np.isclose(_chk["lag_total"], _chk["reversas_cohort"])
print(f"fechamento lag x numerador do coorte: {_chk['ok'].all()}")
if not _chk["ok"].all():
    display(_chk[~_chk["ok"]])

Reversas do coorte por mês de abertura (quantidade):


lag_grp                             M+0    M+1   M+2  M+3+    total
family              mes_compra                                     
Tech T-shirt Gola U 2026-04    1,440.00 602.00 29.00  3.00 2,074.00
                    2026-05    1,037.00 514.00 17.00  1.00 1,569.00
                    2026-06    1,087.00 366.00  5.00  0.00 1,458.00
                    2026-07    1,231.00 279.00  0.00  0.00 1,510.00
The Perfect Top     2026-04    1,950.00 669.00  7.00  3.00 2,629.00
                    2026-05    1,060.00 435.00  9.00  0.00 1,504.00
                    2026-06      700.00 237.00  1.00  0.00   938.00
                    2026-07      961.00 108.00  0.00  0.00 1,069.00

Mesma tabela em % do coorte:


lag_grp                          M+0   M+1  M+2  M+3+
family              mes_compra                       
Tech T-shirt Gola U 2026-04    69.40 29.00 1.40  0.10
                    2026-05    66.10 32.80 1.10  0.10
                    2026-06    74.60 25.10 0.30  0.00
                    2026-07    81.50 18.50 0.00  0.00
The Perfect Top     2026-04    74.20 25.40 0.30  0.10
                    2026-05    70.50 28.90 0.60  0.00
                    2026-06    74.60 25.30 0.10  0.00
                    2026-07    89.90 10.10 0.00  0.00

fechamento lag x numerador do coorte: True


## D.10 Reconciliação inversa: de onde vem o numerador do método legado

Para cada mês de **abertura** da reversa, qual o mês de compra de origem. Com o universo harmonizado, a categoria "pedido fora do universo válido" deixa de existir (cobertura de `todos_pedidos` é 100% no recorte dos dois produtos) — o numerador legado se reconcilia inteiramente com meses de compra `M-0, M-1, M-2, M-3+`.

In [ ]:
_mr2 = df_rev_aud[df_rev_aud["mes_reversa"].isin(MESES_AUD)].copy()
_mr2["origem"] = np.where(_mr2["data_compra"].isna(), "pedido fora do universo válido",
                          np.where(_mr2["lag_meses"] >= 3, "compra M-3+",
                                   "compra M-" + _mr2["lag_meses"].fillna(0).astype(int).astype(str)))
aud_recon = _mr2.pivot_table(index=["family", "mes_reversa"], columns="origem",
                             values="return_quantity", aggfunc="sum", fill_value=0)
aud_recon["total"] = aud_recon.sum(axis=1)
aud_recon_pct = (aud_recon.drop(columns="total").div(aud_recon["total"], axis=0) * 100).round(1)

print("Numerador do método legado por origem da compra (quantidade):")
display(aud_recon)
print("Mesma tabela em % do numerador legado:")
display(aud_recon_pct)

_chk2 = (aud_recon["total"].reset_index().rename(columns={"mes_reversa": "mes", "total": "recon_total"})
         .merge(mensal_prod, on=["family", "mes"]))
_chk2["ok"] = np.isclose(_chk2["recon_total"], _chk2["reversas_criadas"])
print(f"fechamento reconciliação x numerador legado: {_chk2['ok'].all()}")

Numerador do método legado por origem da compra (quantidade):


origem                           compra M-0  compra M-1  compra M-2  \
family              mes_reversa                                       
Tech T-shirt Gola U 2026-04        1,440.00    1,072.00       25.00   
                    2026-05        1,037.00      602.00       37.00   
                    2026-06        1,087.00      514.00       29.00   
                    2026-07        1,231.00      366.00       17.00   
The Perfect Top     2026-04        1,950.00      984.00       26.00   
                    2026-05        1,060.00      669.00       35.00   
                    2026-06          700.00      435.00        7.00   
                    2026-07          961.00      237.00        9.00   

origem                           compra M-3+    total  
family              mes_reversa                        
Tech T-shirt Gola U 2026-04            45.00 2,582.00  
                    2026-05            41.00 1,717.00  
                    2026-06            28.00 1,658.00  
                    2026-07            12.00 1,626.00  
The Perfect Top     2026-04            52.00 3,012.00  
                    2026-05            25.00 1,789.00  
                    2026-06             7.00 1,149.00  
                    2026-07            11.00 1,218.00

Mesma tabela em % do numerador legado:


origem                           compra M-0  compra M-1  compra M-2  \
family              mes_reversa                                       
Tech T-shirt Gola U 2026-04           55.80       41.50        1.00   
                    2026-05           60.40       35.10        2.20   
                    2026-06           65.60       31.00        1.70   
                    2026-07           75.70       22.50        1.00   
The Perfect Top     2026-04           64.70       32.70        0.90   
                    2026-05           59.30       37.40        2.00   
                    2026-06           60.90       37.90        0.60   
                    2026-07           78.90       19.50        0.70   

origem                           compra M-3+  
family              mes_reversa               
Tech T-shirt Gola U 2026-04             1.70  
                    2026-05             2.40  
                    2026-06             1.70  
                    2026-07             0.70  
The Perfect Top     2026-04             1.70  
                    2026-05             1.40  
                    2026-06             0.60  
                    2026-07             0.90

fechamento reconciliação x numerador legado: True


## D.11 Maturação e censura à direita

Curva de referência construída com coortes já maduras (10 a 3 meses antes do início da janela) dos mesmos dois produtos: qual % do numerador lifetime já apareceu após N dias da compra. Aplicada aos coortes da janela para estimar quanto ainda falta e projetar o T&D% do coorte.

`td_pct_cohort_projetado` é **estimativa**, não número oficial — serve só para separar efeito de maturação de efeito de metodologia.

In [ ]:
q_curva_aud = f"""
WITH {CTE_SKUS},
todos_pedidos AS (
  SELECT order_name, MIN(DATE(TIMESTAMP(processed_at),'America/Sao_Paulo')) AS data_compra
  FROM `insider-data-lake.business.insider_orders`
  WHERE order_name IS NOT NULL AND processed_at IS NOT NULL
  GROUP BY 1
),
rev AS (
  SELECT DISTINCT order_name, id_reversa, sku, created_at, SAFE_CAST(return_quantity AS FLOAT64) AS rq
  FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br`
  WHERE status <> 'Cancelado' AND order_name IS NOT NULL AND sku IS NOT NULL
    AND id_reversa IS NOT NULL AND created_at IS NOT NULL
)
SELECT DATE_DIFF(DATE(r.created_at,'America/Sao_Paulo'), tp.data_compra, DAY) AS dias, SUM(r.rq) AS qt
FROM rev r
JOIN skus s ON r.sku = s.sku
JOIN todos_pedidos tp USING (order_name)
WHERE tp.data_compra BETWEEN '{(MES_INI_AUD - 10).start_time.date()}' AND '{(MES_INI_AUD - 3).end_time.date()}'
  AND DATE_DIFF(DATE(r.created_at,'America/Sao_Paulo'), tp.data_compra, DAY) >= 0
GROUP BY 1 ORDER BY 1
"""
aud_curva = client.query(q_curva_aud).to_dataframe(create_bqstorage_client=False)
aud_curva["cum_pct"] = (aud_curva["qt"].cumsum() / aud_curva["qt"].sum() * 100).round(2)
print(f"curva de referência: {aud_curva['qt'].sum():,.0f} reversas de coortes maduras")
for _d in [15, 30, 60, 90, 180]:
    print(f"  até {_d:>3}d após a compra: {aud_curva[aud_curva['dias'] <= _d]['qt'].sum() / aud_curva['qt'].sum():.1%} do lifetime")


def _pct_realizado(dias):
    if dias <= 0:
        return np.nan
    return aud_curva[aud_curva["dias"] <= dias]["qt"].sum() / aud_curva["qt"].sum() * 100


aud_matur = cohort_prod.copy()
aud_matur["meio_mes"] = pd.PeriodIndex(aud_matur["mes"], freq="M").to_timestamp() + pd.Timedelta(days=14)
aud_matur["dias_maturacao"] = (HOJE - aud_matur["meio_mes"]).dt.days
aud_matur["pct_realizado_est"] = aud_matur["dias_maturacao"].apply(_pct_realizado).round(1)
aud_matur["td_pct_cohort_projetado"] = (aud_matur["td_pct_cohort"] / aud_matur["pct_realizado_est"] * 100).round(2)
display(aud_matur[["family", "mes", "dias_maturacao", "td_pct_cohort", "pct_realizado_est", "td_pct_cohort_projetado"]])

curva de referência: 57,874 reversas de coortes maduras
  até  15d após a compra: 81.0% do lifetime
  até  30d após a compra: 95.5% do lifetime
  até  60d após a compra: 98.5% do lifetime
  até  90d após a compra: 99.3% do lifetime
  até 180d após a compra: 99.9% do lifetime


,family,mes,dias_maturacao,td_pct_cohort,pct_realizado_est,td_pct_cohort_projetado
0,Tech T-shirt Gola U,2026-04,112,5.57,99.50,5.60
1,Tech T-shirt Gola U,2026-05,82,5.56,99.10,5.61
2,Tech T-shirt Gola U,2026-06,51,5.81,98.10,5.92
3,Tech T-shirt Gola U,2026-07,21,4.95,90.10,5.49
4,The Perfect Top,2026-04,112,8.46,99.50,8.50
5,The Perfect Top,2026-05,82,6.79,99.10,6.85
6,The Perfect Top,2026-06,51,6.29,98.10,6.41
7,The Perfect Top,2026-07,21,5.53,90.10,6.14


## D.11 Maturação e censura à direita

Curva de referência harmonizada (mesma ancoragem `todos_pedidos`) construída com coortes já maduras (10 a 3 meses antes do início da janela) dos mesmos dois produtos: qual % do numerador lifetime já apareceu após N dias da compra. Aplicada aos coortes da janela para estimar quanto ainda falta e projetar o T&D% do coorte.

`td_pct_cohort_projetado` é **estimativa**, não número oficial — serve só para separar efeito de maturação de efeito de ancoragem.

In [ ]:
QUERY_TD_MENSAL_LEGADO = """
-- Definição ANTIGA de Troca & Devolução — visão mensal por fluxo.
-- NÃO é a fonte da verdade do OKR (a SoT é o coorte de compra, Parte C). Mantida para auditoria.
--
-- Numerador: peças com solicitação de troca/devolução criada no mês.
-- Denominador: peças vendidas no mesmo mês.
-- Não há vínculo de coorte, pedido ou SKU entre vendas e reversas.

WITH params AS (
  SELECT
    DATE '2026-01-01' AS start_date,
    DATE_SUB(
      DATE_TRUNC(CURRENT_DATE('America/Sao_Paulo'), MONTH),
      INTERVAL 1 MONTH
    ) AS ultimo_mes_fechado
),

compras_filtradas AS (
  SELECT DISTINCT
    order_id,
    DATE(
      TIMESTAMP(processed_at),
      'America/Sao_Paulo'
    ) AS data_compra
  FROM `insider-data-lake.business.insider_orders`
  WHERE order_status = 'paid'
    AND is_cancelled = FALSE
    AND (
      coupon_code IS NULL
      OR (
        NOT STARTS_WITH(coupon_code, 'TF-')
        AND NOT STARTS_WITH(coupon_code, 'TFIN')
        AND NOT STARTS_WITH(coupon_code, 'IR')
        AND NOT coupon_code LIKE '%Item errado%'
      )
    )
    AND order_name IS NOT NULL
    AND processed_at IS NOT NULL
    AND store IN (
      'shopify_insider-store-loja',
      'shopify_insider-world'
    )
),

vendas_mensais AS (
  SELECT
    DATE_TRUNC(c.data_compra, MONTH) AS mes_referencia,
    SUM(COALESCE(i.quantity, 0)) AS qt_itens_vendidos
  FROM compras_filtradas AS c
  INNER JOIN `insider-data-lake.business.insider_order_items` AS i
    ON c.order_id = i.order_id
  CROSS JOIN params AS p
  WHERE i.sku IS NOT NULL
    AND c.data_compra >= p.start_date
    AND c.data_compra < DATE_ADD(p.ultimo_mes_fechado, INTERVAL 1 MONTH)
  GROUP BY 1
),

reversas_deduplicadas AS (
  SELECT DISTINCT
    order_name,
    id_reversa,
    sku,
    DATE(created_at, 'America/Sao_Paulo') AS data_reversa,
    SAFE_CAST(return_quantity AS FLOAT64) AS return_quantity
  FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br`
  WHERE status <> 'Cancelado'
    AND order_name IS NOT NULL
    AND sku IS NOT NULL
    AND id_reversa IS NOT NULL
    AND created_at IS NOT NULL
),

reversas_mensais AS (
  SELECT
    DATE_TRUNC(data_reversa, MONTH) AS mes_referencia,
    SUM(COALESCE(return_quantity, 0)) AS qt_itens_revertidos
  FROM reversas_deduplicadas
  CROSS JOIN params AS p
  WHERE data_reversa >= p.start_date
    AND data_reversa < DATE_ADD(p.ultimo_mes_fechado, INTERVAL 1 MONTH)
  GROUP BY 1
),

calendar AS (
  SELECT mes_referencia
  FROM params,
  UNNEST(
    GENERATE_DATE_ARRAY(
      DATE_TRUNC(start_date, MONTH),
      ultimo_mes_fechado,
      INTERVAL 1 MONTH
    )
  ) AS mes_referencia
)

SELECT
  FORMAT_DATE('%Y-%m', c.mes_referencia) AS mes,
  COALESCE(v.qt_itens_vendidos, 0) AS vendas,
  COALESCE(r.qt_itens_revertidos, 0) AS reversas_criadas,
  ROUND(
    SAFE_DIVIDE(
      COALESCE(r.qt_itens_revertidos, 0),
      COALESCE(v.qt_itens_vendidos, 0)
    ) * 100,
    2
  ) AS td_pct_mensal
FROM calendar AS c
LEFT JOIN vendas_mensais AS v USING (mes_referencia)
LEFT JOIN reversas_mensais AS r USING (mes_referencia)
ORDER BY c.mes_referencia
"""

df_legado = client.query(QUERY_TD_MENSAL_LEGADO).to_dataframe(create_bqstorage_client=False)

# coorte oficial da Parte C, agregado por mês (reaproveita df_cohort da extração diária)
_kr = df_cohort.copy()
_kr["mes"] = _kr["dia"].dt.to_period("M").astype(str)
_kr = _kr.groupby("mes").agg(vendas_cohort=("vendas", "sum"), reversas_cohort=("reversas", "sum")).reset_index()
_kr["td_pct_cohort"] = (_kr["reversas_cohort"] / _kr["vendas_cohort"] * 100).round(2)

aud_companhia = df_legado.merge(_kr, on="mes", how="inner")
aud_companhia["delta_pp"] = (aud_companhia["td_pct_mensal"] - aud_companhia["td_pct_cohort"]).round(2)
aud_companhia["delta_rel_pct"] = ((aud_companhia["td_pct_mensal"] / aud_companhia["td_pct_cohort"] - 1) * 100).round(1)
aud_companhia["maduro_em"] = (pd.PeriodIndex(aud_companhia["mes"], freq="M").to_timestamp()
                              + pd.offsets.MonthBegin(1) + pd.Timedelta(days=14))
aud_companhia["status_cohort"] = np.where(HOJE >= aud_companhia["maduro_em"], "oficial (maduro)", "em maturação")

display(aud_companhia[["mes", "vendas", "vendas_cohort", "reversas_cohort", "reversas_criadas",
                       "td_pct_cohort", "td_pct_mensal", "delta_pp", "delta_rel_pct", "status_cohort"]])

_dif_den = (aud_companhia["vendas"] - aud_companhia["vendas_cohort"]).abs().max()
print(f"maior diferença de denominador entre os dois métodos: {_dif_den:,.0f} itens (esperado 0)")

,mes,vendas,vendas_cohort,reversas_cohort,reversas_criadas,td_pct_cohort,td_pct_mensal,delta_pp,delta_rel_pct,status_cohort
0,2026-01,217787,217787,"18,329.00","23,048.00",8.42,10.58,2.16,25.70,oficial (maduro)
1,2026-02,228529,228529,"19,411.00","18,126.00",8.49,7.93,-0.56,-6.60,oficial (maduro)
2,2026-03,376554,376554,"35,137.00","33,481.00",9.33,8.89,-0.44,-4.70,oficial (maduro)
3,2026-04,244750,244750,"18,305.00","22,897.00",7.48,9.36,1.88,25.10,oficial (maduro)
4,2026-05,195610,195610,"14,800.00","15,330.00",7.57,7.84,0.27,3.60,oficial (maduro)
5,2026-06,185498,185498,"13,956.00","15,275.00",7.52,8.23,0.71,9.40,oficial (maduro)
6,2026-07,188788,188788,"12,099.00","14,611.00",6.41,7.74,1.33,20.70,em maturação


maior diferença de denominador entre os dois métodos: 0 itens (esperado 0)


## D.13 Gráficos

In [ ]:
# 1) T&D% por produto: oficial (coorte) vs. legado (mensal)
fig = make_subplots(rows=1, cols=len(PRODUTOS_AUDITORIA), shared_yaxes=True, subplot_titles=PRODUTOS_AUDITORIA)
for _j, _prod in enumerate(PRODUTOS_AUDITORIA, start=1):
    sub = aud_lado[aud_lado["family"] == _prod].sort_values("mes")
    fig.add_trace(go.Scatter(x=sub["mes"], y=sub["td_pct_cohort"], name="oficial (coorte)",
                             mode="lines+markers+text", text=sub["td_pct_cohort"], textposition="top center",
                             line=dict(color="#2B6CB0", width=3), showlegend=(_j == 1)), row=1, col=_j)
    fig.add_trace(go.Scatter(x=sub["mes"], y=sub["td_pct_mensal"], name="legado (mensal)",
                             mode="lines+markers+text", text=sub["td_pct_mensal"], textposition="bottom center",
                             line=dict(color="#DD6B20", width=3, dash="dash"), showlegend=(_j == 1)), row=1, col=_j)
fig.update_yaxes(title_text="T&D %", row=1, col=1)
fig.update_layout(title="T&D% por produto — oficial (coorte) vs. legado (mensal) · universo harmonizado",
                  template="plotly_white", height=430, hovermode="x unified")
fig.show()

# 2) Defasagem do coorte (M+0 / M+1 / M+2 / M+3+)
lag_long = aud_lag_pct.reset_index().melt(id_vars=["family", "mes_compra"], var_name="lag", value_name="pct")
lag_long["rotulo"] = lag_long["family"] + " · " + lag_long["mes_compra"]
fig = px.bar(lag_long, x="pct", y="rotulo", color="lag", orientation="h",
             category_orders={"lag": _ordem_lag},
             color_discrete_sequence=["#2B6CB0", "#DD6B20", "#38A169", "#718096"],
             title="Em que mês a reversa do coorte foi aberta (% do coorte)",
             labels={"pct": "% do coorte", "rotulo": "", "lag": "defasagem"})
fig.update_layout(template="plotly_white", height=460, barmode="stack")
fig.show()

# 3) Companhia
fig = go.Figure()
fig.add_trace(go.Scatter(x=aud_companhia["mes"], y=aud_companhia["td_pct_cohort"], name="oficial (coorte)",
                         mode="lines+markers+text", text=aud_companhia["td_pct_cohort"], textposition="top center",
                         line=dict(color="#2B6CB0", width=3)))
fig.add_trace(go.Scatter(x=aud_companhia["mes"], y=aud_companhia["td_pct_mensal"], name="legado (mensal)",
                         mode="lines+markers+text", text=aud_companhia["td_pct_mensal"], textposition="bottom center",
                         line=dict(color="#DD6B20", width=3, dash="dash")))
fig.update_layout(title="Companhia — T&D% oficial (coorte) vs. legado (mensal) · universo harmonizado",
                  yaxis_title="T&D %", template="plotly_white", height=420, hovermode="x unified")
fig.show()

## D.14 Export dos CSVs da auditoria

In [ ]:
df_skus_aud.to_csv(f"{OUT_DIR}/auditoria_td_skus_{DATA_TAG}.csv", index=False)
aud_lado.to_csv(f"{OUT_DIR}/auditoria_td_produto_mes_{DATA_TAG}.csv", index=False)
aud_consol.to_csv(f"{OUT_DIR}/auditoria_td_consolidado_{DATA_TAG}.csv", index=False)
aud_tempo_compra.to_csv(f"{OUT_DIR}/auditoria_td_tempo_compra_{DATA_TAG}.csv")
aud_tempo_entrega.to_csv(f"{OUT_DIR}/auditoria_td_tempo_entrega_{DATA_TAG}.csv")
aud_lag_qt.to_csv(f"{OUT_DIR}/auditoria_td_lag_coorte_{DATA_TAG}.csv")
aud_recon.to_csv(f"{OUT_DIR}/auditoria_td_reconciliacao_{DATA_TAG}.csv")
aud_matur.to_csv(f"{OUT_DIR}/auditoria_td_maturacao_{DATA_TAG}.csv", index=False)
aud_companhia.to_csv(f"{OUT_DIR}/auditoria_td_companhia_{DATA_TAG}.csv", index=False)
df_rev_aud.to_csv(f"{OUT_DIR}/auditoria_td_base_reversas_{DATA_TAG}.csv", index=False)
print(f"CSVs da auditoria exportados em {OUT_DIR} com sufixo {DATA_TAG}")

CSVs da auditoria exportados em ../../outputs com sufixo 20260805


## D.15 Sumário executivo da auditoria

*Execução: 2026-08-04 · coortes de compra 2026-04 a 2026-07 · escopo núcleo (`integrated.skus`, `family`) · 569 SKUs · cobertura do dicionário 100%.*

**Revisão de 2026-08-04:** o universo de reversas do coorte foi harmonizado com o da definição legada — as duas agora somam **exatamente o mesmo conjunto de reversas não canceladas**, ancoradas por `order_name` contra qualquer pedido (não mais restrito a `paid`) e sem exigir SKU casado ao item comprado. A única diferença estrutural remanescente entre as duas é a **ancoragem de data** (mês de compra vs. mês de criação da reversa).

**1. Com o universo harmonizado, o delta caiu de "2–5 vezes" para um viés pequeno e explicável — mudança de ~3–5 p.p. para ~0,5–1,4 p.p.**

| Produto | Mês | Vendas | T&D% oficial (coorte) | T&D% legado (mensal) | Δ p.p. | Δ relativo |
|---|---|---|---|---|---|---|
| Tech T-shirt Gola U | 2026-04 | 37.258 | 5,56 | 6,91 | +1,35 | +24% |
| Tech T-shirt Gola U | 2026-05 | 28.231 | 5,55 | 6,07 | +0,52 | +9% |
| Tech T-shirt Gola U | 2026-06 | 25.084 | 5,80 | 6,59 | +0,79 | +14% |
| Tech T-shirt Gola U | 2026-07 | 30.543 | 4,63 | 5,35 | +0,72 | +16% |
| The Perfect Top | 2026-04 | 31.070 | 8,46 | 9,69 | +1,23 | +15% |
| The Perfect Top | 2026-05 | 22.146 | 6,79 | 8,08 | +1,29 | +19% |
| The Perfect Top | 2026-06 | 14.929 | 6,28 | 7,70 | +1,42 | +23% |
| The Perfect Top | 2026-07 | 19.380 | 5,37 | 6,29 | +0,92 | +17% |

Na companhia o delta muda de sinal mês a mês — 8,42% vs. 10,58% em janeiro (+2,16 p.p.), mas **-0,56 p.p.** em fevereiro e **-0,44 p.p.** em março, voltando a +1,87 p.p. em abril. Essa oscilação de sinal é a assinatura de um efeito puramente temporal (volume de vendas variando mês a mês, deslocado pela defasagem), não de uma assimetria estrutural de universo — que antes empurrava o delta sempre para o mesmo lado (+2,17 a +5,14 p.p., nunca negativo).

**2. A causa agora é só a defasagem entre compra e abertura da reversa — não sobra nenhuma diferença de universo.**
Todas as checagens de fechamento internas confirmam: a soma do numerador por `M+0/M+1/M+2/M+3+` (D.9) reproduz exatamente o numerador do coorte; a reconciliação inversa por mês de compra de origem (D.10) reproduz exatamente o numerador legado; e a categoria "pedido fora do universo válido" **desapareceu** de D.10 — 100% do numerador legado agora se decompõe em `compra M-0, M-1, M-2, M-3+`. O denominador continua idêntico nos dois métodos (diferença de 0 itens).

**3. A defasagem real é curta e cabe quase toda em dois meses — igual ao que já era verdade antes da harmonização.**
Mediana de **7 a 8 dias** entre compra e abertura da reversa; **1 a 2 dias** entre entrega e reversa. Por coorte, **66% a 92%** das reversas abrem no mesmo mês da compra (`M+0`), **8% a 33%** no mês seguinte (`M+1`), ≤1,5% depois disso. O coorte fecha praticamente em M+1 — o buffer de maturação até o dia 15 de M+1 usado no KR continua adequado.

**4. Maturação explica uma fração pequena do delta remanescente.**
Pela curva de referência harmonizada (57.868 reversas de coortes maduras dos mesmos produtos): 81% do numerador lifetime aparece em 15 dias, 95,5% em 30 dias, 99,3% em 90 dias. O coorte de julho está ~89% realizado; projetado, iria de 4,63% para ~5,20% (Gola U) e de 5,37% para ~6,03% (Perfect Top) — o que por si só já fecha boa parte do delta de julho (+0,72 e +0,92 p.p.) sem precisar invocar nenhuma diferença de metodologia.

**5. Recomendação.**
- **Manter o coorte como SoT do OKR**, agora com a query harmonizada (célula da Parte C). É a única das duas definições que responde "qual a qualidade do lote vendido no mês", e as duas passaram a contar exatamente as mesmas reversas.
- **A comparação entre os dois métodos deixou de ser um problema de dados e passou a ser só um efeito de calendário.** Se algum consumidor da métrica legada notar números "maiores" que o KR, a explicação correta agora é defasagem/maturação — não mais duplo padrão de validade de pedido.
- **Dívida herdada (mantida):** a harmonização manteve a decisão de atribuir a reversa por `order_name` (não mais `order_name + sku`). Isso alinha o coorte à definição legada por completo, mas ambas passam a aceitar, em tese, um SKU revertido que não conste como item comprado naquele pedido especificamente — situação residual e hoje medida como 0 casos em D.4 para os dois produtos auditados, mas vale reavaliar se o escopo dos produtos auditados mudar.

### Como validar / reproduzir
Rodar a Parte C (célula 31) e a Parte D inteira. As checagens internas (fechamento do lag contra o numerador do coorte, fechamento da reconciliação contra o numerador legado, identidade dos denominadores, cobertura do dicionário de SKU) imprimem `True`/100% quando tudo fecha.